# Regime-Aware Core-Satellite on Investable PCA Portfolios

This notebook develops a regime-aware core-satellite allocation framework around an investable PCA equity core built on US technology equities.

## What this notebook does
1. Downloads and prepares market data
2. Builds an implied-volatility-based stress score and ML-selected regimes
3. Constructs investable principal portfolios and evaluates their regime-dependent behaviour
4. Builds walk-forward PCA portfolios
5. Screens a multi-asset satellite universe in a portfolio-aware way
6. Tests alternative core-satellite allocation architectures, including:
   - static tangency satellite
   - non-linear outer allocation from PC1 to the satellite
   - linear Tangency → risk-free inner rotation
   - non-linear outer + inner rotation
7. Selects candidate specifications on validation and evaluates them in frozen OOS with paired stationary bootstrap

## Notebook conventions
- **PC1** is the PCA equity core
- **SI** is the continuous stress-intensity signal
- all strategy comparisons are reported **net of Corwin–Schultz transaction-cost adjustments** where applicable
- optional exploratory / diagnostic cells have been removed from this cleaned version to keep the notebook repository-ready


## 0) Setup and reusable helpers

In [ ]:
# -------------------------
# PARAMETERS
# -------------------------
from yfinance import download
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


plt.rcParams["figure.figsize"] = (10, 4)
pd.options.display.float_format = "{:,.6f}".format

# Example single economy universe (US Tech)
TICKERS_US_TECH = ["AAPL","MSFT","GOOGL","AMZN","QCOM","CSCO","NVDA","ORCL","TXN","ADBE","IBM","CRM","AMAT","INTU"]

# Global regime variable (use ONE series consistently across all economies)
IV_TICKER_PRIMARY = "^VXN"   # Nasdaq-100 implied vol
IV_TICKER_FALLBACK = "^VIX"  # fallback

# Benchmark used only for supervised regime calibration target
BENCH_TICKER = "QQQ"         # market benchmark for 21/22d forward downside target

# Historical risk-free proxy
RF_TICKER_PRIMARY = "^IRX"   # 13-week Treasury bill annualized yield proxy (Yahoo Finance)
RF_ANN = 252                 # trading-day annualization used for daily return conversion

START = "2012-01-01"
END   = "2026-04-30"

TRAIN_END_DEFAULT = "2017-12-31"
TEST_START_DEFAULT = "2018-01-01"

# Regime-state construction
REGIME_FREQ = "D"           # "D" or "W" (weekly)
SMOOTH_WINDOW = 5           # e.g. 5 trading days smoothing for log IV
EWMA_LAMBDA = 0.94          # for z-score of log IV (optional)
REGIME_MIN_OBS = 252        # min observations before live expanding-percentile labelling starts

# ML-calibrated regime selection
# State: IV score observed at t.
# Target: QQQ entry-to-trough forward drawdown over the next H trading days.
REGIME_TARGET_H = 22        # use 21 or 22; 22 ~= one trading month
REGIME_MIN_REGIMES = 2
REGIME_MAX_REGIMES = 5
REGIME_MIN_LEAF_FRAC = 0.15 # keeps regimes economically material; avoids tiny leaves
REGIME_CV_SPLITS = 5        # TimeSeriesSplit folds for tree model selection
REGIME_TREE_RANDOM_STATE = 42
REGIME_USE_EXPANDING_PERCENTILES = True

# Fallback only used if ML calibration cannot run because too few observations are available.
REGIME_FALLBACK_Q = (0.33, 0.66)

# PCA settings
K_BAR = 7                   # number of PCs to build portfolios for

# Portfolio normalisation
GROSS_NORM = 1.0            # enforce sum(abs(w)) = 1.0

# Walk-forward settings (optional)
WF_TRAIN_YEARS = 5          # rolling training window length
WF_STEP_MONTHS = 3          # rebalance / re-estimation frequency
WF_K_CHOICE = 7             # PCs in walk-forward (keep fixed for simplicity)

# Transaction cost model (used only in "tradable strategy" overlays)
TC_BPS = 10                 # bps per 1.0 turnover (gross)


In [ ]:
# ============================================================
# HELPERS (keep everything reusable here)
# ============================================================

def linear_returns_from_prices(adj_close: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    out = adj_close.pct_change()
    if isinstance(out, pd.DataFrame):
        return out.dropna(how="all")
    return out.dropna()

def ewma_mean_std(x: pd.Series, lam: float = 0.94) -> pd.DataFrame:
    x = x.dropna().astype(float)
    m = np.zeros(len(x))
    v = np.zeros(len(x))
    m[0] = x.iloc[0]
    v[0] = 0.0
    for t in range(1, len(x)):
        m[t] = lam * m[t-1] + (1 - lam) * x.iloc[t]
        innov = x.iloc[t] - m[t-1]
        v[t] = lam * v[t-1] + (1 - lam) * innov**2
    s = np.sqrt(np.maximum(v, 1e-12))
    return pd.DataFrame({"ewma_mean": m, "ewma_std": s}, index=x.index)

def build_state_z_from_iv(iv_level: pd.Series,
                          freq: str = "D",
                          smooth_window: int = 5,
                          lam: float = 0.94,
                          use_zscore: bool = True) -> pd.Series:
    # Build z_t from implied vol level:
    #   x_t = log(IV_t)
    #   smooth(x_t) = rolling mean over past 'smooth_window' periods (incl. today)
    #   z_t = (smooth - ewma_mean) / ewma_std   (optional)
    iv = iv_level.dropna().astype(float)
    if freq.upper().startswith("W"):
        iv = iv.resample("W-FRI").last()  # align to week close
    x = np.log(iv)
    x_smooth = x.rolling(window=smooth_window, min_periods=smooth_window).mean().dropna()

    if not use_zscore:
        z = x_smooth.copy()
        z.name = "state_z"
        return z

    stats = ewma_mean_std(x_smooth, lam=lam)
    z = (x_smooth.loc[stats.index] - stats["ewma_mean"]) / stats["ewma_std"]
    z.name = "state_z"
    return z


def _canonical_regime_names(k: int) -> list[str]:
    """Ordered names from low-risk/low-IV to high-risk/high-IV."""
    name_map = {
        1: ["ALL"],
        2: ["LOW", "HIGH"],
        3: ["LOW", "MID", "HIGH"],
        4: ["LOW", "MID_LOW", "MID_HIGH", "HIGH"],
        5: ["LOW", "LOW_MID", "MID", "MID_HIGH", "HIGH"],
    }
    if k in name_map:
        return name_map[k]
    return [f"R{i+1}" for i in range(k)]


def _ordered_regime_names(reg: pd.Series, regime_order: list[str] | None = None) -> list[str]:
    vals = [x for x in pd.Series(reg).dropna().unique().tolist()]
    if regime_order is not None:
        return [x for x in regime_order if x in vals]
    preferred = ["ALL", "LOW", "LOW_MID", "MID", "MID_LOW", "MID_HIGH", "HIGH"]
    out = [x for x in preferred if x in vals]
    out += sorted([x for x in vals if x not in out])
    return out


def _assign_labels_from_cutoffs(z: pd.Series,
                                cutoffs: list[float] | np.ndarray,
                                labels: list[str] | None = None) -> pd.Series:
    z = pd.Series(z).dropna().astype(float)
    cutoffs = np.asarray(cutoffs, dtype=float)
    cutoffs = np.sort(cutoffs[np.isfinite(cutoffs)])
    k = len(cutoffs) + 1
    labels = labels or _canonical_regime_names(k)
    if len(labels) != k:
        raise ValueError(f"labels length {len(labels)} must equal number of regimes {k}.")
    bins = np.searchsorted(cutoffs, z.values, side="right")
    return pd.Series([labels[int(i)] for i in bins], index=z.index, dtype="object")


def regime_labels_from_state(z: pd.Series,
                             q=(0.33, 0.66),
                             expanding: bool = True,
                             min_obs: int = 252,
                             labels: list[str] | None = None) -> pd.Series:
    """
    Non-leaky regime labels from a continuous state score via quantile cutoffs.

    q can contain any number of sorted quantile cutoffs, so both the number of
    regimes and the boundaries can be calibrated outside this function.
    """
    z = pd.Series(z).dropna().astype(float)
    q = tuple(float(x) for x in q)
    if len(q) == 0:
        out = pd.Series(index=z.index, data="ALL", dtype="object")
        return out
    if any((x <= 0 or x >= 1) for x in q) or any(np.diff(q) <= 0):
        raise ValueError("q must be strictly increasing and inside (0, 1).")

    k = len(q) + 1
    labels = labels or _canonical_regime_names(k)
    if len(labels) != k:
        raise ValueError(f"labels length {len(labels)} must equal number of regimes {k}.")

    if not expanding:
        cutoffs = z.quantile(q).values
        return _assign_labels_from_cutoffs(z, cutoffs=cutoffs, labels=labels)

    out = pd.Series(index=z.index, dtype="object")
    for i in range(len(z)):
        if i + 1 < min_obs:
            continue
        hist = z.iloc[: i + 1]
        cutoffs = hist.quantile(q).values
        bin_id = int(np.searchsorted(cutoffs, z.iloc[i], side="right"))
        out.iloc[i] = labels[bin_id]
    return out.dropna()


def forward_entry_drawdown(r: pd.Series, horizon: int = 22) -> pd.DataFrame:
    """
    Forward entry-to-trough drawdown over the next H trading days.

    At date t, the statistic is min cumulative return from t+1 to t+H,
    measured relative to today's entry wealth. It is known only ex-post and
    should therefore be used as a calibration label, not as a live input.
    """
    r = pd.Series(r).dropna().astype(float)
    idx = pd.to_datetime(r.index)
    r.index = idx
    vals = []
    ends = []
    out_idx = []
    for i in range(0, len(r) - horizon):
        window = r.iloc[i + 1 : i + 1 + horizon]
        if len(window) < horizon:
            continue
        wealth = (1.0 + window).cumprod()
        vals.append(float(wealth.min() - 1.0))
        ends.append(window.index[-1])
        out_idx.append(r.index[i])
    return pd.DataFrame(
        {
            f"fwd_entry_drawdown_{horizon}d": vals,
            f"target_end_{horizon}d": pd.to_datetime(ends),
        },
        index=pd.DatetimeIndex(out_idx),
    )


def _tree_cutoffs_1d(tree_model) -> list[float]:
    """Extract sorted split thresholds from a fitted univariate sklearn tree."""
    thresholds = tree_model.tree_.threshold
    cutoffs = sorted(float(x) for x in thresholds if np.isfinite(x) and x != -2.0)
    # sklearn can create repeated numerical thresholds in edge cases.
    out = []
    for x in cutoffs:
        if not out or abs(x - out[-1]) > 1e-10:
            out.append(x)
    return out


def select_regime_spec_tree(regime_df: pd.DataFrame,
                            state_col: str,
                            target_col: str,
                            train_end: str | pd.Timestamp,
                            target_end_col: str | None = None,
                            min_regimes: int = 2,
                            max_regimes: int = 5,
                            min_leaf_frac: float = 0.15,
                            cv_splits: int = 5,
                            random_state: int = 42,
                            fallback_q: tuple[float, ...] = (0.33, 0.66),
                            use_one_se_rule: bool = True) -> dict:
    """
    Select number of regimes and IV-score boundaries via a supervised ML model.

    Model: univariate DecisionTreeRegressor.
    X: IV score observed at t.
    y: positive forward downside severity, i.e. -forward_drawdown_{t+1:t+H}.

    The model is selected on development data only via TimeSeriesSplit.
    Rows whose target window ends after train_end are excluded to prevent
    target leakage across the development/OOS boundary.
    """
    try:
        from sklearn.tree import DecisionTreeRegressor
        from sklearn.model_selection import TimeSeriesSplit
        from sklearn.metrics import mean_squared_error
    except Exception as exc:
        q = tuple(fallback_q)
        return {
            "method": "fallback_tertiles_no_sklearn",
            "error": repr(exc),
            "n_regimes": len(q) + 1,
            "z_cutoffs": [],
            "quantile_cutoffs": q,
            "labels": _canonical_regime_names(len(q) + 1),
            "cv_table": pd.DataFrame(),
            "dev_regime_stats": pd.DataFrame(),
        }

    df = regime_df.copy()
    df = df[[c for c in [state_col, target_col, target_end_col] if c is not None and c in df.columns]].dropna()
    train_end = pd.Timestamp(train_end)
    dev = df.loc[df.index <= train_end].copy()
    if target_end_col is not None and target_end_col in dev.columns:
        dev = dev.loc[pd.to_datetime(dev[target_end_col]) <= train_end].copy()

    if len(dev) < 250:
        q = tuple(fallback_q)
        return {
            "method": "fallback_tertiles_too_few_obs",
            "n_regimes": len(q) + 1,
            "z_cutoffs": [],
            "quantile_cutoffs": q,
            "labels": _canonical_regime_names(len(q) + 1),
            "cv_table": pd.DataFrame(),
            "dev_regime_stats": pd.DataFrame(),
            "n_dev": int(len(dev)),
        }

    X = dev[[state_col]].astype(float).values
    y = (-dev[target_col].astype(float)).clip(lower=0.0).values

    n = len(dev)
    min_leaf_abs = max(20, int(np.ceil(min_leaf_frac * n)))
    max_possible_leaves = max(2, int(np.floor(n / max(min_leaf_abs, 1))))
    k_grid = [k for k in range(int(min_regimes), int(max_regimes) + 1) if k <= max_possible_leaves]
    if not k_grid:
        k_grid = [2]

    n_splits = min(int(cv_splits), max(2, n // max(min_leaf_abs, 1) - 1))
    n_splits = max(2, n_splits)

    rows = []
    for k in k_grid:
        fold_mse = []
        fold_leaves = []
        tscv = TimeSeriesSplit(n_splits=n_splits)
        for train_idx, val_idx in tscv.split(X):
            # Make the leaf constraint relative to the CV training fold to avoid impossible splits.
            fold_min_leaf = max(10, int(np.ceil(min_leaf_frac * len(train_idx))))
            model = DecisionTreeRegressor(
                max_leaf_nodes=k,
                min_samples_leaf=fold_min_leaf,
                random_state=random_state,
            )
            model.fit(X[train_idx], y[train_idx])
            pred = model.predict(X[val_idx])
            fold_mse.append(mean_squared_error(y[val_idx], pred))
            fold_leaves.append(model.get_n_leaves())
        rows.append({
            "requested_max_regimes": int(k),
            "cv_mse_mean": float(np.mean(fold_mse)),
            "cv_mse_std": float(np.std(fold_mse, ddof=1)) if len(fold_mse) > 1 else 0.0,
            "cv_mse_se": float(np.std(fold_mse, ddof=1) / np.sqrt(len(fold_mse))) if len(fold_mse) > 1 else 0.0,
            "avg_realized_leaves": float(np.mean(fold_leaves)),
            "min_leaf_abs_final": int(min_leaf_abs),
            "n_dev": int(n),
        })

    cv_table = pd.DataFrame(rows).sort_values("requested_max_regimes").reset_index(drop=True)
    best_row = cv_table.loc[cv_table["cv_mse_mean"].idxmin()]
    if use_one_se_rule:
        threshold = best_row["cv_mse_mean"] + best_row["cv_mse_se"]
        selected_row = cv_table.loc[cv_table["cv_mse_mean"] <= threshold].sort_values("requested_max_regimes").iloc[0]
    else:
        selected_row = best_row

    selected_k = int(selected_row["requested_max_regimes"])
    final_model = DecisionTreeRegressor(
        max_leaf_nodes=selected_k,
        min_samples_leaf=min_leaf_abs,
        random_state=random_state,
    )
    final_model.fit(X, y)

    z_cutoffs = _tree_cutoffs_1d(final_model)
    actual_k = len(z_cutoffs) + 1
    labels = _canonical_regime_names(actual_k)

    # Convert ML z-thresholds into development-sample percentile cutoffs.
    # These are then usable with the original non-leaky expanding-quantile machinery.
    z_dev = dev[state_col].astype(float)
    q_cutoffs = tuple(float((z_dev <= c).mean()) for c in z_cutoffs)
    q_cutoffs = tuple(float(np.clip(q, 0.01, 0.99)) for q in q_cutoffs)

    dev_reg = _assign_labels_from_cutoffs(z_dev, cutoffs=z_cutoffs, labels=labels)
    stats = (
        pd.DataFrame({
            "regime": dev_reg,
            "iv_score": z_dev,
            "fwd_drawdown": dev[target_col].astype(float),
            "downside_severity": (-dev[target_col].astype(float)).clip(lower=0.0),
        })
        .groupby("regime")
        .agg(
            n=("fwd_drawdown", "size"),
            iv_score_mean=("iv_score", "mean"),
            iv_score_min=("iv_score", "min"),
            iv_score_max=("iv_score", "max"),
            mean_fwd_drawdown=("fwd_drawdown", "mean"),
            median_fwd_drawdown=("fwd_drawdown", "median"),
            mean_downside_severity=("downside_severity", "mean"),
            p95_downside_severity=("downside_severity", lambda x: float(np.quantile(x, 0.95))),
        )
    )
    stats["freq"] = stats["n"] / stats["n"].sum()
    stats = stats.reindex(labels)

    return {
        "method": "DecisionTreeRegressor_TimeSeriesSplit_one_se" if use_one_se_rule else "DecisionTreeRegressor_TimeSeriesSplit_best_mse",
        "state_col": state_col,
        "target_col": target_col,
        "target_end_col": target_end_col,
        "train_end": str(train_end.date()),
        "n_dev": int(n),
        "n_regimes": int(actual_k),
        "requested_max_regimes": int(selected_k),
        "z_cutoffs": tuple(float(x) for x in z_cutoffs),
        "quantile_cutoffs": q_cutoffs,
        "labels": labels,
        "cv_table": cv_table,
        "dev_regime_stats": stats,
        "model": final_model,
    }


def apply_regime_spec(z: pd.Series,
                      regime_spec: dict,
                      expanding_percentiles: bool = True,
                      min_obs: int = 252) -> pd.Series:
    """Apply the frozen ML-selected regime specification to a live/OOS state score."""
    labels = regime_spec.get("labels")
    if expanding_percentiles:
        q = regime_spec.get("quantile_cutoffs", ())
        return regime_labels_from_state(z, q=q, expanding=True, min_obs=min_obs, labels=labels)
    return _assign_labels_from_cutoffs(z, cutoffs=regime_spec.get("z_cutoffs", ()), labels=labels)


def weighted_mean_cov(X: pd.DataFrame, p: pd.Series | None = None):
    # Mean/cov with optional probabilities p (must sum to 1 on index).
    if p is None:
        mu = X.mean(axis=0).values
        Xc = X.values - mu.reshape(1, -1)
        Sigma = (Xc.T @ Xc) / X.shape[0]
        return pd.Series(mu, index=X.columns, name="mu"), pd.DataFrame(Sigma, index=X.columns, columns=X.columns)

    X = X.loc[p.index].copy()
    w = p.values.reshape(-1, 1)
    mu = (w * X.values).sum(axis=0)
    Xc = X.values - mu.reshape(1, -1)
    Sigma = (w * Xc).T @ Xc
    return pd.Series(mu, index=X.columns, name="mu"), pd.DataFrame(Sigma, index=X.columns, columns=X.columns)

def correlation_pca(Sigma: np.ndarray):
    # Correlation PCA: rho = D^{-1} Sigma D^{-1}.
    Sigma = np.asarray(Sigma, dtype=float)
    vol = np.sqrt(np.clip(np.diag(Sigma), 1e-18, None))
    Dinv = np.diag(1.0 / vol)
    rho = Dinv @ Sigma @ Dinv
    lam, V = np.linalg.eigh(rho)   # ascending
    lam = lam[::-1]
    V = V[:, ::-1]

    # Sign convention: largest abs element in each eigenvector positive
    max_abs_row = np.argmax(np.abs(V), axis=0)
    s = np.sign(V[max_abs_row, np.arange(V.shape[0])])
    s[s == 0] = 1.0
    V = V * s.reshape(1, -1)
    return lam, V, vol, rho

def pc_portfolio_weights_from_corr_pca(V: np.ndarray, vol: np.ndarray, gross_norm: float = 1.0) -> pd.DataFrame:
    # Investible PC portfolios: w_k ∝ D^{-1} v_k, then normalize to sum(abs(w))=gross_norm.
    V = np.asarray(V, dtype=float)
    vol = np.asarray(vol, dtype=float)
    Dinv = 1.0 / vol
    W = V * Dinv.reshape(-1, 1)
    gross = np.sum(np.abs(W), axis=0)
    gross[gross == 0] = 1.0
    W = W / gross.reshape(1, -1) * gross_norm
    return pd.DataFrame(W)

def factor_returns(R: pd.DataFrame, W: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(R.values @ W.values, index=R.index, columns=[f"PC{i+1}" for i in range(W.shape[1])])

# ---------- Basic utilities ----------
def max_drawdown_from_wealth(wealth: pd.Series) -> float:
    wealth = wealth.dropna().astype(float)
    peak = wealth.cummax()
    dd = wealth/peak - 1.0
    return float(dd.min()) if len(dd) else np.nan

def ulcer_index(r: pd.Series) -> float:
    r = r.dropna().astype(float)
    if len(r)==0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth/peak - 1.0
    return float(np.sqrt(np.mean(dd**2)))

def cagr(r: pd.Series, ann: int = 252) -> float:
    r = r.dropna().astype(float)
    if len(r)==0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    years = len(r)/ann
    return float(wealth.iloc[-1]**(1/years)-1) if years>0 else np.nan

def annualized_yield_pct_to_daily_return(y_ann_pct: pd.Series, ann: int = 252) -> pd.Series:
    """
    Convert an annualized yield quoted in percent (e.g. Yahoo ^IRX) into
    an approximate daily simple return using trading-day compounding.
    """
    y = pd.Series(y_ann_pct).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    y = (y / 100.0).clip(lower=-0.999999)
    rf_daily = (1.0 + y) ** (1.0 / ann) - 1.0
    rf_daily.name = "rf_daily"
    return rf_daily

def _coerce_rf_daily(r: pd.Series, rf_daily: float | pd.Series | None = 0.0) -> pd.Series:
    if rf_daily is None:
        return pd.Series(0.0, index=r.index, dtype=float, name="rf_daily")
    if np.isscalar(rf_daily):
        return pd.Series(float(rf_daily), index=r.index, dtype=float, name="rf_daily")
    rf = pd.Series(rf_daily, dtype=float).reindex(r.index).ffill().fillna(0.0)
    rf.name = getattr(rf_daily, "name", "rf_daily")
    return rf

# ---------- Performance metrics (historical RF-aware for Sharpe / Sortino / Martin) ----------
def perf_metrics(r: pd.Series, rf_daily: float | pd.Series | None = 0.0, ann: int = 252) -> dict:
    r = r.dropna().astype(float)
    if len(r) < 5:
        return {"n": int(len(r))}

    rf_s = _coerce_rf_daily(r, rf_daily)
    ex = r - rf_s

    mu_d = ex.mean()
    vol_d = ex.std(ddof=1)
    shr = (mu_d/vol_d)*np.sqrt(ann) if vol_d>0 else np.nan

    downside = ex.copy()
    downside[downside > 0] = 0.0
    dvol = downside.std(ddof=1)
    sor = (mu_d/dvol)*np.sqrt(ann) if dvol>0 else np.nan

    cagr_ = cagr(r, ann=ann)
    cagr_excess = cagr(ex, ann=ann) if (1.0 + ex).min() > 0 else np.nan
    wealth = (1.0 + r).cumprod()
    mxdd = max_drawdown_from_wealth(wealth)
    calmar = (cagr_/abs(mxdd)) if (mxdd < 0 and np.isfinite(mxdd)) else np.nan

    ui = ulcer_index(r)
    rf_cagr = cagr(rf_s, ann=ann)
    martin = ((cagr_ - rf_cagr)/ui) if (ui>0 and np.isfinite(ui)) else np.nan

    return {
        "n": int(len(r)),
        "CAGR": float(cagr_),
        "CAGR_excess": float(cagr_excess) if np.isfinite(cagr_excess) else np.nan,
        "RF_CAGR": float(rf_cagr) if np.isfinite(rf_cagr) else np.nan,
        "MxDD": float(mxdd),
        "ShR": float(shr),
        "SoR": float(sor),
        "Calmar": float(calmar),
        "Ulcer": float(ui),
        "Martin": float(martin),
    }

# ---------- Regimes ----------

def regime_from_state(z: pd.Series,
                      q: tuple[float, ...] | list[float] = (0.33, 0.66),
                      cutoffs: tuple[float, ...] | list[float] | None = None,
                      labels: list[str] | None = None) -> pd.Series:
    """Convenience regime labeller using either fixed z-cutoffs or full-sample quantiles."""
    z = pd.Series(z).dropna().astype(float)
    if cutoffs is None:
        cutoffs = z.quantile(tuple(q)).values if len(tuple(q)) else []
    return _assign_labels_from_cutoffs(z, cutoffs=cutoffs, labels=labels)

# ---------- Metrics tables (overall + conditional) ----------
def compute_metrics_by_regime(F: pd.DataFrame,
                              reg: pd.Series,
                              rf_daily: float | pd.Series | None = 0.0,
                              ann: int = 252,
                              regime_order: list[str] | None = None) -> dict:
    reg = reg.reindex(F.index).dropna()
    F = F.loc[reg.index]
    out = {"overall": {c: perf_metrics(F[c], rf_daily=rf_daily, ann=ann) for c in F.columns}}
    for rn in _ordered_regime_names(reg, regime_order=regime_order):
        idx = reg[reg == rn].index
        rf_sub = rf_daily.loc[idx] if hasattr(rf_daily, "loc") else rf_daily
        out[rn] = {c: perf_metrics(F.loc[idx, c], rf_daily=rf_sub, ann=ann) for c in F.columns}
    return out

def table_from_metrics(mdict: dict, key: str) -> pd.DataFrame:
    rows=[]
    for regime_name, d in mdict.items():
        for pc, mets in d.items():
            rows.append({"regime": regime_name, "PC": pc, key: mets.get(key, np.nan)})
    return pd.DataFrame(rows).pivot(index="PC", columns="regime", values=key)

# ---------- Turnover / similarity ----------
def turnover(w_prev: np.ndarray, w_now: np.ndarray) -> float:
    w_prev = np.asarray(w_prev, float).ravel()
    w_now  = np.asarray(w_now, float).ravel()
    return float(np.sum(np.abs(w_now - w_prev)))

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, float).ravel()
    b = np.asarray(b, float).ravel()
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na==0 or nb==0:
        return np.nan
    return float(np.dot(a,b)/(na*nb))

def align_signs_to_previous(W_prev: np.ndarray, W_now: np.ndarray) -> np.ndarray:
    # column-wise sign alignment using cosine similarity
    W_now = W_now.copy()
    for j in range(W_now.shape[1]):
        if cosine_sim(W_prev[:,j], W_now[:,j]) < 0:
            W_now[:,j] *= -1
    return W_now

def stability_from_W_hist(W_hist: list, pcs: list) -> pd.DataFrame:
    if len(W_hist) < 2:
        return pd.DataFrame()
    rows=[]
    for i in range(1, len(W_hist)):
        W_prev = W_hist[i-1]["W"][pcs].values
        W_now  = W_hist[i]["W"][pcs].values
        W_now  = align_signs_to_previous(W_prev, W_now)
        row={"rebalance": W_hist[i]["rebalance"]}
        for j,pc in enumerate(pcs):
            row[f"cos_{pc}"] = cosine_sim(W_prev[:,j], W_now[:,j])
            row[f"to_{pc}"]  = turnover(W_prev[:,j], W_now[:,j])
        rows.append(row)
    return pd.DataFrame(rows).set_index("rebalance")

# ---------- Politis–Romano stationary bootstrap ----------
def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator) -> np.ndarray:
    # Politis & Romano: block length geometric with mean avg_block_len
    p = 1.0/avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t-1] + 1) % T
    return idx

def stationary_bootstrap_df(df: pd.DataFrame, n_boot: int = 1000, avg_block_len: float = 10.0, seed: int = 1):
    rng = np.random.default_rng(seed)
    T = len(df)
    for _ in range(n_boot):
        ii = stationary_bootstrap_indices(T, avg_block_len, rng)
        yield df.iloc[ii].reset_index(drop=True)

def summarize_vec(x: np.ndarray) -> pd.Series:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x)==0:
        return pd.Series({"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0})
    return pd.Series({
        "mean": float(np.mean(x)),
        "p05": float(np.quantile(x, 0.05)),
        "p50": float(np.quantile(x, 0.50)),
        "p95": float(np.quantile(x, 0.95)),
        "n": int(len(x))
    })


def bootstrap_metric_tables(df_joint: pd.DataFrame,
                           pc_cols: list,
                           z_col: str = "z",
                           regime_q=(0.33, 0.66),
                           regime_labels: list[str] | None = None,
                           metrics=("CAGR","ShR","SoR","Calmar","Martin","MxDD"),
                           n_boot: int = 1000,
                           avg_block_len: float = 10.0,
                           seed: int = 1,
                           rf_daily: float | pd.Series | None = 0.0,
                           ann: int = 252) -> dict:
    # returns dict[(metric, regime)] -> table PCs x {mean,p05,p50,p95,n}
    out = {}
    sims = stationary_bootstrap_df(df_joint, n_boot=n_boot, avg_block_len=avg_block_len, seed=seed)
    regime_labels = regime_labels or _canonical_regime_names(len(tuple(regime_q)) + 1)
    regime_names = ["overall"] + list(regime_labels)
    mats = {(m, rn): np.full((n_boot, len(pc_cols)), np.nan) for m in metrics for rn in regime_names}
    for i, sim in enumerate(sims):
        z = sim[z_col]
        reg = regime_from_state(z, q=regime_q, labels=regime_labels)
        F = sim[pc_cols]
        md = compute_metrics_by_regime(F, reg, rf_daily=rf_daily, ann=ann, regime_order=regime_labels)
        for rn in regime_names:
            for j, pc in enumerate(pc_cols):
                for m in metrics:
                    mats[(m, rn)][i, j] = md.get(rn, {}).get(pc, {}).get(m, np.nan)
    for m in metrics:
        for rn in regime_names:
            rows = []
            for j, pc in enumerate(pc_cols):
                ss = summarize_vec(mats[(m, rn)][:, j]); ss["PC"] = pc
                rows.append(ss)
            out[(m, rn)] = pd.DataFrame(rows).set_index("PC").sort_values("mean", ascending=False)
    return out


# -------------------------
# Stationary Bootstrap (Politis–Romano)
# -------------------------
def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator):
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t-1] + 1) % T
    return idx

def stationary_bootstrap_series(df: pd.DataFrame, n_boot: int = 2000, avg_block_len: float = 10.0, seed: int = 0):
    rng = np.random.default_rng(seed)
    T = len(df)
    out = []
    for _ in range(n_boot):
        idx = stationary_bootstrap_indices(T, avg_block_len, rng)
        out.append(df.iloc[idx].reset_index(drop=True))
    return out

# -------------------------
# Politis–White style automatic block length (data-driven)
# -------------------------

def _autocov(x: np.ndarray, lag: int) -> float:
    """Sample autocovariance at lag (unbiased-ish)."""
    x = np.asarray(x, float)
    n = len(x)
    if lag >= n:
        return np.nan
    x0 = x[: n - lag]
    x1 = x[lag:]
    return float(np.mean((x0 - x0.mean()) * (x1 - x1.mean())))

def _long_run_var_bartlett(x: np.ndarray, L: int) -> float:

    g0 = _autocov(x, 0)
    s = g0
    for k in range(1, L + 1):
        wk = 1.0 - k / (L + 1.0)
        gk = _autocov(x, k)
        s += 2.0 * wk * gk
    return float(max(s, 1e-18))

def politis_white_block_length(
    s: pd.Series,
    use_abs: bool = True,
    max_lag: int | None = None,
    clip: tuple[int, int] = (2, 252),
) -> int:

    x = s.dropna().astype(float).values
    if use_abs:
        x = np.abs(x)

    T = len(x)
    if T < 50:
        # Too short: return something conservative
        return int(np.clip(10, *clip))

    # Choose truncation lag for long-run variance estimation (rule of thumb)
    if max_lag is None:
        max_lag = int(np.floor(T ** (1 / 3)))
        max_lag = max(5, min(max_lag, 200))

    # Long-run variance (captures dependence)
    lrv = _long_run_var_bartlett(x, L=max_lag)
    var = float(np.var(x, ddof=1))
    var = max(var, 1e-18)

    # Heuristic Politis–White style scaling:
    # L* ∝ (lrv/var)^(2/3) * T^(1/3)
    # (constant absorbed; in practice we calibrate gently with c=1.0)
    ratio = max(lrv / var, 1e-6)
    L_star = int(np.round((ratio ** (2 / 3)) * (T ** (1 / 3))))

    return int(np.clip(L_star, *clip))

# -------------------------
# Choose block length by matching volatility clustering (ACF)
# -------------------------

def _acf(x: np.ndarray, nlags: int) -> np.ndarray:

    x = np.asarray(x, float)
    x = x - np.mean(x)
    n = len(x)
    denom = np.dot(x, x)
    if denom <= 0:
        return np.zeros(nlags + 1)
    acf = np.empty(nlags + 1, float)
    acf[0] = 1.0
    for k in range(1, nlags + 1):
        acf[k] = np.dot(x[:-k], x[k:]) / denom
    return acf

def choose_block_length_by_acf_matching(
    s: pd.Series,
    candidates: list[int] = [5, 10, 20],
    nlags: int = 20,
    n_boot: int = 300,
    use_abs: bool = True,
    seed: int = 1,
    stationary_bootstrap_indices_fn=None,
    distance: str = "l2",
) -> dict:

    if stationary_bootstrap_indices_fn is None:
        raise ValueError("Pass your helper stationary_bootstrap_indices_fn (Politis–Romano indices generator).")

    r = s.dropna().astype(float)
    x = np.abs(r.values) if use_abs else r.values
    T = len(x)
    if T < nlags + 10:
        raise ValueError("Series too short for requested nlags.")

    target = _acf(x, nlags=nlags)

    rng = np.random.default_rng(seed)
    scores = {}
    boot_means = {}

    for L in candidates:
        acfs = np.zeros((n_boot, nlags + 1))
        for b in range(n_boot):
            ii = stationary_bootstrap_indices_fn(T, float(L), rng)
            xb = x[ii]
            acfs[b, :] = _acf(xb, nlags=nlags)
        m = acfs.mean(axis=0)
        boot_means[L] = m

        diff = m[1:] - target[1:]  # ignore lag0
        if distance == "l1":
            score = float(np.mean(np.abs(diff)))
        elif distance == "l2":
            score = float(np.sqrt(np.mean(diff**2)))
        elif distance == "weighted":
            # weight shorter lags more (more relevant for clustering)
            w = 1.0 / np.arange(1, nlags + 1)
            score = float(np.sqrt(np.mean((diff**2) * w)))
        else:
            raise ValueError("distance must be 'l1', 'l2', or 'weighted'.")

        scores[L] = score

    scores_ser = pd.Series(scores).sort_index()
    best_L = int(scores_ser.idxmin())

    return {
        "best_L": best_L,
        "scores": scores_ser,
        "target_acf": target,
        "boot_acf_mean": boot_means,
        "use_abs": use_abs,
        "nlags": nlags,
        "n_boot": n_boot,
        "distance": distance
    }

# -------------------------
# Compute and table regime-based risk & performance metrics
# -------------------------


def compute_metrics_from_joint(df: pd.DataFrame,
                               regime_q: tuple[float, ...] | None = None,
                               regime_labels: list[str] | None = None):
    z = df["z"]
    rf = df["rf"] if "rf" in df.columns else 0.0
    if regime_q is None:
        regime_q = globals().get("REGIME_Q", (0.33, 0.66))
    if regime_labels is None:
        regime_labels = globals().get("REGIME_ORDER", _canonical_regime_names(len(tuple(regime_q)) + 1))
    reg = regime_from_state(z, q=regime_q, labels=regime_labels)
    out = {"overall": {}}
    for rn in regime_labels:
        out[rn] = {}
    for pc in PCS:
        out["overall"][pc] = perf_metrics(df[pc], rf_daily=rf)
        for rn in regime_labels:
            idx = reg[reg == rn].index
            rf_sub = rf.loc[idx] if hasattr(rf, "loc") else rf
            out[rn][pc] = perf_metrics(df.loc[idx, pc], rf_daily=rf_sub)
    return out


def summarize_vec(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x)==0:
        return {"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0}
    return {"mean": float(np.mean(x)),
            "p05": float(np.quantile(x, 0.05)),
            "p50": float(np.quantile(x, 0.50)),
            "p95": float(np.quantile(x, 0.95)),
            "n": int(len(x))}

def bootstrap_ci_table(metric_key: str, regime_name: str):
    mat = np.full((N_BOOT, len(PCS)), np.nan)
    for i, sim in enumerate(sims):
        res = compute_metrics_from_joint(sim)
        for j, pc in enumerate(PCS):
            mat[i, j] = res[regime_name][pc].get(metric_key, np.nan)
    rows=[]
    for j, pc in enumerate(PCS):
        s = summarize_vec(mat[:,j]); s["PC"]=pc
        rows.append(s)
    return pd.DataFrame(rows).set_index("PC").sort_values("mean", ascending=False)

# -------------------------
# Walk-forward PCA
# -------------------------
def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (ts + pd.offsets.DateOffset(months=m)).normalize()

def walk_forward_pca(R: pd.DataFrame,
                     start_test: str,
                     train_years: int = 5,
                     step_months: int = 3,
                     k: int = 7,
                     gross_norm: float = 1.0,
                     tc_bps: float = 0.0):
    R = R.dropna()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    weights_hist = []
    f_all = []
    costs_all = []
    w_prev = None

    reb = t0
    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        mu, Sigma = weighted_mean_cov(R_train)
        lam, V, vol, rho = correlation_pca(Sigma.values)
        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

        f = factor_returns(R_test, W)
        f_all.append(f)

        if tc_bps > 0.0:
            w_now = W.values
            if w_prev is None:
                c = np.zeros(W.shape[1])
            else:
                to = np.array([turnover(w_prev[:, j], w_now[:, j]) for j in range(w_now.shape[1])])
                c = (tc_bps / 1e4) * to
            seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)
            seg_cost.iloc[0, :] = c
            costs_all.append(seg_cost)
            w_prev = w_now
        else:
            w_prev = W.values

        weights_hist.append({"rebalance": reb, "train_start": train_start, "train_end": train_end, "W": W})
        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)
    F_net = F - C.reindex(F.index).fillna(0.0)

    return F, F_net, weights_hist


## 1) Data download & returns

In [ ]:
all_tickers = sorted(set(TICKERS_US_TECH + [IV_TICKER_PRIMARY, IV_TICKER_FALLBACK, BENCH_TICKER]))
px = download(all_tickers, start=START, end=END, auto_adjust=False, progress=False).dropna(how="all")

px_adj_close_all = px["Adj Close"][TICKERS_US_TECH].copy()
px_high = px["High"][TICKERS_US_TECH].copy()
px_low = px["Low"][TICKERS_US_TECH].copy()

# For PCA we still want a common-support panel.
px_stocks = px_adj_close_all.dropna(how="any")

px_adj = px["Adj Close"].copy()
iv = px_adj[IV_TICKER_PRIMARY] if IV_TICKER_PRIMARY in px_adj.columns else None
if iv is None or iv.dropna().empty:
    iv = px_adj[IV_TICKER_FALLBACK]
iv.name = "IV"

R = linear_returns_from_prices(px_stocks)

# Benchmark used for supervised regime calibration: QQQ forward downside.
px_benchmark = px_adj[BENCH_TICKER].dropna().astype(float).rename(BENCH_TICKER)
B_ret = linear_returns_from_prices(px_benchmark)
if isinstance(B_ret, pd.DataFrame):
    B_ret = B_ret.iloc[:, 0]
B_ret = pd.Series(B_ret, index=pd.to_datetime(B_ret.index), name=f"{BENCH_TICKER}_ret").dropna().astype(float)

# -------------------------
# Historical risk-free proxy
# -------------------------
rf_raw = download(RF_TICKER_PRIMARY, start=START, end=END, auto_adjust=False, progress=False)

def _extract_single_close(x):
    if isinstance(x, pd.Series):
        return x.astype(float)
    if not isinstance(x, pd.DataFrame):
        return pd.Series(dtype=float)
    for field in ["Adj Close", "Close"]:
        if field in x.columns:
            s = x[field]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            return s.astype(float)
    if x.shape[1] == 1:
        return x.iloc[:, 0].astype(float)
    return pd.Series(dtype=float)

rf_ann_pct = _extract_single_close(rf_raw).dropna()
if rf_ann_pct.empty:
    print(f"WARNING: no historical risk-free proxy downloaded from {RF_TICKER_PRIMARY}; using zero daily RF.")
    rf_daily_hist = pd.Series(0.0, index=R.index, name="rf_daily")
    rf_source_used = "ZERO_FALLBACK"
else:
    rf_daily_hist = annualized_yield_pct_to_daily_return(rf_ann_pct, ann=RF_ANN)
    rf_daily_hist = rf_daily_hist.reindex(R.index).ffill().fillna(0.0)
    rf_source_used = RF_TICKER_PRIMARY

print("R shape:", R.shape)
print(f"Benchmark for regime calibration: {BENCH_TICKER} | return obs: {len(B_ret):,}")
print(
    f"RF source: {rf_source_used} | "
    f"non-null daily obs on R index: {int(rf_daily_hist.notna().sum())} | "
    f"sample mean daily RF (bps): {1e4 * rf_daily_hist.mean():.4f}"
)
R.head()


## 2) Build IV score and ML-calibrated non-leaky regime labels

In [ ]:
# ------------------------------------------------------------------
# 2.1 IV score observed at t
# ------------------------------------------------------------------
z = build_state_z_from_iv(
    iv,
    freq=REGIME_FREQ,
    smooth_window=SMOOTH_WINDOW,
    lam=EWMA_LAMBDA,
    use_zscore=True,
)
z_daily = z.reindex(R.index).ffill().dropna()
z_daily.name = "iv_score"

# ------------------------------------------------------------------
# 2.2 Ex-post calibration target: QQQ entry-to-trough forward drawdown
#     over the next H trading days. This is a historical label only.
# ------------------------------------------------------------------
qqq_fwd_dd = forward_entry_drawdown(B_ret, horizon=REGIME_TARGET_H)
target_col = f"fwd_entry_drawdown_{REGIME_TARGET_H}d"
target_end_col = f"target_end_{REGIME_TARGET_H}d"

regime_calibration_df = pd.concat(
    [
        z_daily.rename("iv_score"),
        qqq_fwd_dd[[target_col, target_end_col]],
    ],
    axis=1,
).dropna()

# ------------------------------------------------------------------
# 2.3 ML selection of number of regimes and percentile cutoffs
#     Train/development only; target windows crossing TRAIN_END are excluded.
# ------------------------------------------------------------------
REGIME_SPEC = select_regime_spec_tree(
    regime_df=regime_calibration_df,
    state_col="iv_score",
    target_col=target_col,
    target_end_col=target_end_col,
    train_end=TRAIN_END_DEFAULT,
    min_regimes=REGIME_MIN_REGIMES,
    max_regimes=REGIME_MAX_REGIMES,
    min_leaf_frac=REGIME_MIN_LEAF_FRAC,
    cv_splits=REGIME_CV_SPLITS,
    random_state=REGIME_TREE_RANDOM_STATE,
    fallback_q=REGIME_FALLBACK_Q,
    use_one_se_rule=True,
)

REGIME_Q = tuple(REGIME_SPEC["quantile_cutoffs"])
REGIME_ORDER = list(REGIME_SPEC["labels"])

print("ML-calibrated regime specification")
print("method:", REGIME_SPEC.get("method"))
print("development rows used:", REGIME_SPEC.get("n_dev"))
print("selected number of regimes K:", REGIME_SPEC.get("n_regimes"))
print("selected IV-score cutoffs:", REGIME_SPEC.get("z_cutoffs"))
print("selected percentile cutoffs:", REGIME_Q)
print("regime order:", REGIME_ORDER)

if not REGIME_SPEC.get("cv_table", pd.DataFrame()).empty:
    print("\nTime-series CV table used to select K")
    display(REGIME_SPEC["cv_table"])

if not REGIME_SPEC.get("dev_regime_stats", pd.DataFrame()).empty:
    print(f"\nDevelopment-sample regime diagnostics versus {BENCH_TICKER} {REGIME_TARGET_H}d forward downside")
    display(REGIME_SPEC["dev_regime_stats"])

# ------------------------------------------------------------------
# 2.4 Live/OOS-implementable labelling
#     Default: use ML-selected percentile levels with expanding quantiles.
#     This keeps the original non-leaky percentile machinery while removing
#     hard-coded 33/66 and hard-coded K=3.
# ------------------------------------------------------------------
reg = apply_regime_spec(
    z_daily,
    REGIME_SPEC,
    expanding_percentiles=REGIME_USE_EXPANDING_PERCENTILES,
    min_obs=REGIME_MIN_OBS,
)

print("\nRegime counts on full aligned sample after min_obs filter:")
print(reg.value_counts().reindex(REGIME_ORDER))

z_daily.loc[reg.index].plot(title="IV score $z_t$ used for live regime assignment")
plt.show()

tmp = pd.DataFrame({"iv_score": z_daily.loc[reg.index], "regime": reg})
tmp["regime_code"] = tmp["regime"].map({name: i for i, name in enumerate(REGIME_ORDER)})
tmp["regime_code"].plot(title="ML-selected regime code (ordered low-to-high IV/downside state)")
plt.show()

## 3) Static PCA (train) → factor portfolios → conditional performance (test)

In [ ]:
train = R.loc[:TRAIN_END_DEFAULT].dropna()
test  = R.loc[TEST_START_DEFAULT:].dropna()

mu_hat, Sigma_hat = weighted_mean_cov(train)
lam, V, vol, rho = correlation_pca(Sigma_hat.values)

W = pc_portfolio_weights_from_corr_pca(V[:, :K_BAR], vol, gross_norm=GROSS_NORM)
W.index = R.columns
W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

F_test = factor_returns(test, W)

reg_test = reg.reindex(F_test.index).dropna()
F_test = F_test.loc[reg_test.index]
rf_test = rf_daily_hist.reindex(F_test.index).ffill().fillna(0.0)

metrics_overall = {c: perf_metrics(F_test[c], rf_daily=rf_test) for c in F_test.columns}

def metrics_table(d: dict, key: str):
    return pd.Series({k: v.get(key, np.nan) for k,v in d.items()})

summary = pd.DataFrame({
    "CAGR": metrics_table(metrics_overall, "CAGR"),
    "CAGR_excess": metrics_table(metrics_overall, "CAGR_excess"),
    "RF_CAGR": metrics_table(metrics_overall, "RF_CAGR"),
    "ShR":  metrics_table(metrics_overall, "ShR"),
    "SoR":  metrics_table(metrics_overall, "SoR"),
    "Martin": metrics_table(metrics_overall, "Martin"),
    "MxDD": metrics_table(metrics_overall, "MxDD"),
    "Calmar": metrics_table(metrics_overall, "Calmar"),
})
print(f"Static test metrics: Sharpe / Sortino / Martin use historical RF proxy {rf_source_used}.")
summary


In [ ]:
((1+F_test.iloc[:, :7]).cumprod()).plot(title="Test Equity Curves — PC portfolios (static PCA)")
plt.show()

for rn in REGIME_ORDER:
    idx = reg_test[reg_test == rn].index
    cols = F_test.columns[:7]
    ((1+F_test.loc[idx, cols]).cumprod()).plot(title=f"Equity Curves within {rn} regime (subset days)")
    plt.show()


### Bootstrap block-length calibration
Use a liquid benchmark (QQQ) to choose a reasonable stationary-bootstrap block length before running the uncertainty analysis.


In [ ]:
# ============================================================
# Bootstrap block-length calibration (QQQ benchmark)
# ============================================================

# B_ret is defined in the data-download cell from BENCH_TICKER.
if "B_ret" not in globals() or pd.Series(B_ret).dropna().empty:
    px_B = download(BENCH_TICKER, start=START, end=END, auto_adjust=False, progress=False)["Adj Close"].dropna(how="all")
    B_ret = linear_returns_from_prices(px_B)
    if isinstance(B_ret, pd.DataFrame):
        B_ret = B_ret.iloc[:, 0]
    B_ret = pd.Series(B_ret).dropna().astype(float)

# Pick an automatic block length on |QQQ returns| to capture volatility clustering.
L_pw = politis_white_block_length(B_ret, use_abs=True, clip=(2, 126))
print("Politis-White-style block length:", L_pw)

# For the rest of the notebook, use the automatic Politis-White-style value by default.
AVG_BLOCK = L_pw


## 4) Stationary bootstrap (Politis–Romano) for uncertainty of metrics
Bootstrap jointly (PC returns + state variable) by resampling time indices, preserving dependence.


In [ ]:
# ============================================================
# Bootstrap CIs — ALL selected PCs (first 7) and key metrics
# (Politis–Romano stationary bootstrap)
# ============================================================

PCS = [c for c in F_test.columns if c.startswith("PC")][:7]
KEYS = ["CAGR", "CAGR_excess", "ShR", "SoR", "Calmar", "Martin", "MxDD"]

df_joint = pd.concat(
    [
        F_test[PCS],
        z_daily.reindex(F_test.index).rename("z"),
        rf_daily_hist.reindex(F_test.index).rename("rf"),
    ],
    axis=1,
).dropna()
print("Joint test shape:", df_joint.shape)
print(f"Bootstrap metrics use historical RF proxy {rf_source_used} for Sharpe / Sortino / Martin and excess CAGR.")

N_BOOT = 1000
AVG_BLOCK = L_pw
sims = stationary_bootstrap_series(df_joint.reset_index(drop=True), n_boot=N_BOOT, avg_block_len=AVG_BLOCK, seed=1)

for key in KEYS:
    print("\n====================", key, "====================")
    for regime_name in ["overall"] + REGIME_ORDER:
        display(bootstrap_ci_table(key, regime_name).head(10))


## 5) Walk-forward PCA and implementation-aware evaluation


In [ ]:
# ============================================================
# Corwin–Schultz helpers (single asset + panel + one-way cost)
# ============================================================

def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    """
    Corwin–Schultz (2012) bid–ask spread estimator from daily high/low prices.
    Returns the FULL proportional spread, e.g. 0.01 = 1%.
    Input must be two aligned Series for ONE asset.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)

    idx = high.dropna().index.intersection(low.dropna().index)
    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()

    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)

    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)

    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = 0.10,
) -> pd.DataFrame:
    """
    Apply Corwin–Schultz asset by asset on a panel of highs/lows.
    Returns a DataFrame indexed by date, columns=tickers, with FULL spreads.
    """
    px_high = px_high.copy()
    px_low = px_low.copy()

    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()

        idx = h.index.intersection(l.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])

        if clip_upper is not None:
            s = s.clip(upper=clip_upper)

        out[t] = s

    panel = pd.DataFrame(out).sort_index()
    panel = panel.reindex(columns=tickers)
    return panel


def cs_panel_to_one_way_cost(cs_spreads: pd.DataFrame) -> pd.DataFrame:
    """
    Convert FULL spread to ONE-WAY implementation cost.
    If spread = ask-bid over mid, a one-way execution cost is half-spread.
    """
    return 0.5 * cs_spreads.astype(float)

In [ ]:
# ============================================================
# Walk-forward PCA with:
# - sign alignment vs previous rebalance
# - flat bps OR asset-level time-varying costs
# - cost charged on first realized return of each segment
# ============================================================

def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (pd.Timestamp(ts) + pd.offsets.DateOffset(months=m)).normalize()


def _align_W_to_prev(W_prev: pd.DataFrame | None, W_now: pd.DataFrame) -> pd.DataFrame:
    """
    Align PC signs to previous rebalance so that sign flips do not create fake turnover.
    """
    if W_prev is None:
        return W_now.copy()

    W_aligned = W_now.copy()
    common_cols = [c for c in W_now.columns if c in W_prev.columns]

    for c in common_cols:
        a = W_prev[c].values.astype(float)
        b = W_now[c].values.astype(float)

        if np.dot(a, b) < 0:
            W_aligned[c] = -W_aligned[c]

    return W_aligned


def walk_forward_pca(
    R: pd.DataFrame,
    start_test: str,
    train_years: int = 5,
    step_months: int = 3,
    k: int = 7,
    gross_norm: float = 1.0,
    tc_bps: float | None = 0.0,
    asset_costs: pd.DataFrame | None = None,
    charge_initial_rebalance: bool = False,
):
    """
    Walk-forward PCA with either:
      1) flat linear transaction cost in bps via tc_bps
      2) asset-level time-varying one-way costs via asset_costs (date x asset)

    Cost of each PC at rebalance:
        cost_pc_t = sum_i one_way_cost_{i,t} * |Δw_{i,t}|

    Cost is charged on the first realized return in each out-of-sample segment.
    """

    R = R.dropna().sort_index().copy()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    if asset_costs is not None:
        asset_costs = (
            asset_costs.reindex(index=R.index, columns=R.columns)
            .sort_index()
            .ffill()
            .fillna(0.0)
            .astype(float)
        )

    weights_hist = []
    f_all = []
    costs_all = []
    W_prev = None

    reb = t0
    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        mu, Sigma = weighted_mean_cov(R_train)
        lam, V, vol, rho = correlation_pca(Sigma.values)

        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]

        # Critical: align signs to previous rebalance BEFORE turnover/costs/returns comparison
        W = _align_W_to_prev(W_prev, W)

        f = factor_returns(R_test, W)
        f_all.append(f)

        seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)

        if charge_initial_rebalance or (W_prev is not None):
            dW = W.abs() if W_prev is None else (W - W_prev).abs()

            # first ACTUAL trading day in the segment
            trade_date = R_test.index[0]

            if asset_costs is not None:
                c_assets = asset_costs.loc[trade_date, R.columns]
                c_pc = dW.mul(c_assets, axis=0).sum(axis=0)
            else:
                c = 0.0 if tc_bps is None else float(tc_bps) / 1e4
                c_pc = dW.sum(axis=0) * c

            seg_cost.iloc[0, :] = c_pc.reindex(f.columns).values

        costs_all.append(seg_cost)
        W_prev = W.copy()

        weights_hist.append(
            {
                "rebalance": reb,
                "train_start": train_start,
                "train_end": train_end,
                "W": W.copy(),
            }
        )

        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)

    C = C.reindex(index=F.index, columns=F.columns).fillna(0.0)
    F_net = F - C

    return F, F_net, weights_hist

In [ ]:
# ============================================================
# Time-varying transaction costs from Corwin–Schultz spreads
# ============================================================

# Full bid-ask spreads estimated asset by asset
cs_spreads = corwin_schultz_panel(
    px_high=px_high,
    px_low=px_low,
    tickers=list(R.columns),
    clip_upper=None,
)

# Convert full spread to one-way implementation cost
asset_costs = (
    cs_panel_to_one_way_cost(cs_spreads)
    .reindex(index=R.index, columns=R.columns)
    .ffill()
    .fillna(0.0)
)

print("One-way cost summary (bps) by asset:")
display((1e4 * asset_costs).describe().T[["mean", "50%", "max"]].sort_values("50%"))

F_wf, F_wf_net, W_hist = walk_forward_pca(
    R=R,
    start_test=TEST_START_DEFAULT,
    train_years=WF_TRAIN_YEARS,
    step_months=WF_STEP_MONTHS,
    k=WF_K_CHOICE,
    gross_norm=GROSS_NORM,
    tc_bps=None,
    asset_costs=asset_costs,
    charge_initial_rebalance=False,
)

print("WF factor returns shape:", F_wf.shape, " | net:", F_wf_net.shape)

PCS_WF = [c for c in F_wf.columns if c.startswith("PC")][:7]

if not F_wf.empty:
    ((1 + F_wf[PCS_WF]).cumprod()).plot(title="Walk-forward equity curves (gross, before costs)")
    plt.show()

    ((1 + F_wf_net[PCS_WF]).cumprod()).plot(
        title="Walk-forward equity curves (net of Corwin–Schultz one-way costs)"
    )
    plt.show()

    reg_wf = reg.reindex(F_wf.index)
    rf_wf = rf_daily_hist.reindex(F_wf.index).ffill().fillna(0.0)

    m_wf_g = {"overall": {c: perf_metrics(F_wf[c], rf_daily=rf_wf) for c in PCS_WF}}
    m_wf_n = {"overall": {c: perf_metrics(F_wf_net[c], rf_daily=rf_wf) for c in PCS_WF}}

    for rn in REGIME_ORDER:
        idx = reg_wf[reg_wf == rn].index
        m_wf_g[rn] = {c: perf_metrics(F_wf.loc[idx, c], rf_daily=rf_wf) for c in PCS_WF}
        m_wf_n[rn] = {c: perf_metrics(F_wf_net.loc[idx, c], rf_daily=rf_wf) for c in PCS_WF}

    print(f"Walk-forward metrics: Sharpe / Sortino / Martin use historical RF proxy {rf_source_used}.")
    for key in ["CAGR", "CAGR_excess", "ShR", "SoR", "Calmar", "Martin", "MxDD"]:
        print("\n---", key, "(gross) ---")
        display(table_from_metrics(m_wf_g, key))

        print("---", key, "(net of CS costs) ---")
        display(table_from_metrics(m_wf_n, key))


## 6) V3 — IV-aware downside overlay on PC1

This section upgrades the project from regime-aware static / walk-forward PCA to a **maturity-matched, IV-aware downside overlay** on the core **PC1** strategy.

The logic is:

1. use **today’s implied-volatility information** (`log IV`, `Δ log IV`, rolling `z`-score of `log IV`) as predictors;
2. define future downside as the **worst forward return relative to today’s entry wealth** over the next `H` trading days;
3. test whether IV contains useful information about that downside target;
4. translate the signal into a **systematic exposure-scaling overlay** on walk-forward **net** PC1 returns.

Important implementation choices:

- the downside target is **non-leaky** in the overlay backtest: when forecasting at date `t`, the model is trained only on targets whose forward horizon is already fully realized;
- the exposure signal is observed at **close `t`** and is applied from the **next daily return onward**;
- the base strategy remains the walk-forward **net-of-spread-costs** PC1 series computed above.


In [ ]:

# ============================================================
# V3 helpers: downside target, IV predictors, exploratory tests,
# and expanding OOS downside overlay on PC1
# ============================================================

def forward_worst_return_from_returns(r: pd.Series, horizon: int = 21) -> pd.Series:
    """
    Worst forward cumulative return relative to today's entry wealth:
        Y_t = min_{1 <= h <= H} (W_{t+h} / W_t - 1)
    where W is the wealth index implied by returns r.
    """
    r = pd.Series(r).dropna().astype(float)
    wealth = (1.0 + r).cumprod()
    w = wealth.values.astype(float)

    out = np.full(len(w), np.nan)
    for i in range(len(w)):
        j = min(len(w), i + horizon + 1)
        if i + 1 >= j:
            continue
        out[i] = np.min(w[i + 1:j] / w[i] - 1.0)

    s = pd.Series(out, index=wealth.index, name=f"worst_fwd_ret_{horizon}d")
    return s


def build_iv_predictor_panel(
    iv_level: pd.Series,
    target_index: pd.Index,
    z_window: int = 252,
    z_min_obs: int = 126,
) -> pd.DataFrame:
    """
    Build minimal IV predictors aligned to a target strategy index.
    Predictors are observed at date t (close):
      - log_iv
      - dlog_iv
      - log_iv_z  (rolling z-score, non-leaky because it only uses data up to t)
    """
    iv = (
        pd.Series(iv_level)
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
        .reindex(target_index)
        .ffill()
    )

    log_iv = np.log(iv)
    dlog_iv = log_iv.diff()

    roll_mean = log_iv.rolling(z_window, min_periods=z_min_obs).mean()
    roll_std = log_iv.rolling(z_window, min_periods=z_min_obs).std(ddof=1).replace(0.0, np.nan)
    log_iv_z = (log_iv - roll_mean) / roll_std

    out = pd.DataFrame(
        {
            "log_iv": log_iv,
            "dlog_iv": dlog_iv,
            "log_iv_z": log_iv_z,
        },
        index=target_index,
    )
    return out


def bucket_summary(
    df: pd.DataFrame,
    feature: str,
    target: str,
    event: str,
    q: int = 5,
) -> pd.DataFrame:
    """
    Exploratory monotonicity check by feature quantile bucket.
    """
    tmp = df[[feature, target, event]].dropna().copy()
    if len(tmp) == 0:
        return pd.DataFrame()

    tmp["bucket"] = pd.qcut(tmp[feature], q=q, duplicates="drop")
    out = tmp.groupby("bucket", observed=False).agg(
        n=(target, "size"),
        feature_mean=(feature, "mean"),
        target_mean=(target, "mean"),
        target_median=(target, "median"),
        target_p10=(target, lambda x: x.quantile(0.10)),
        event_rate=(event, "mean"),
    )
    return out


def hac_ols_summary(
    y: pd.Series,
    X: pd.DataFrame,
    hac_lags: int = 21,
):
    """
    OLS with HAC (Newey-West) standard errors for overlapping forward targets.
    """
    try:
        import statsmodels.api as sm
    except Exception as e:
        print("statsmodels not available for HAC OLS:", e)
        return None, pd.DataFrame()

    df = pd.concat([pd.Series(y, name="y"), X], axis=1).dropna()
    if len(df) < 25:
        return None, pd.DataFrame()

    y_ = df["y"].astype(float)
    X_ = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")
    res = sm.OLS(y_, X_).fit(cov_type="HAC", cov_kwds={"maxlags": int(hac_lags)})

    table = pd.DataFrame(
        {
            "coef": res.params,
            "tstat_HAC": res.tvalues,
            "pvalue_HAC": res.pvalues,
        }
    )
    return res, table


def _standardize_train_now(X_train: pd.DataFrame, x_now: pd.Series):
    """
    Standardize predictors using training-sample moments only.
    """
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0, ddof=0).replace(0.0, 1.0)
    Xs = (X_train - mu) / sd
    xs = (pd.Series(x_now, index=X_train.columns) - mu) / sd
    return Xs, xs


def _fit_predict_logit_prob(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    x_now: pd.Series,
) -> float:
    """
    Robust probability forecast for downside event:
    prefers sklearn logistic regression; falls back to statsmodels GLM.
    """
    X_train = pd.DataFrame(X_train).astype(float)
    y_train = pd.Series(y_train).astype(int)
    x_now = pd.Series(x_now, index=X_train.columns).astype(float)

    Xs, xs = _standardize_train_now(X_train, x_now)

    if y_train.nunique() < 2:
        return float(y_train.mean())

    try:
        from sklearn.linear_model import LogisticRegression

        clf = LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
        )
        clf.fit(Xs.values, y_train.values)
        p = clf.predict_proba(xs.values.reshape(1, -1))[0, 1]
        return float(p)

    except Exception:
        try:
            import statsmodels.api as sm

            Xc = sm.add_constant(Xs, has_constant="add")
            xc = sm.add_constant(xs.to_frame().T, has_constant="add")
            model = sm.GLM(y_train.values, Xc.values, family=sm.families.Binomial())
            res = model.fit()
            p = float(res.predict(xc.values)[0])
            return float(p)
        except Exception:
            return float(y_train.mean())


def prob_to_exposure(
    p: float,
    exposure_map: list[tuple[float, float]] | None = None,
) -> float:
    """
    Map downside-event probability to a gross exposure multiplier.
    exposure_map is a sorted list of (probability_cap, exposure).
    """
    if exposure_map is None:
        exposure_map = [
            (0.10, 1.00),
            (0.20, 0.75),
            (0.35, 0.50),
            (np.inf, 0.25),
        ]

    if not np.isfinite(p):
        return 1.0

    for cap, exp_ in exposure_map:
        if p <= cap:
            return float(exp_)
    return float(exposure_map[-1][1])


def expanding_downside_overlay(
    base_r: pd.Series,
    X: pd.DataFrame,
    downside_target: pd.Series,
    downside_threshold: float = -0.05,
    horizon: int = 21,
    min_train: int = 252,
    refit_every: int = 21,
    exposure_map: list[tuple[float, float]] | None = None,
    overlay_cost_bps: float = 0.0,
):
    """
    Expanding-window OOS overlay:
    - at date t, estimate P(Y_t < threshold | X_t) using only observations s
      whose forward target is fully known by t (non-leaky training end = t-horizon)
    - map that probability into an exposure multiplier
    - apply the exposure from the NEXT daily return onward

    Returns a dict with:
      overlay_returns, probability, exposure_signal, exposure_applied, diagnostics
    """
    base_r = pd.Series(base_r).dropna().astype(float)
    X = pd.DataFrame(X).reindex(base_r.index)
    y = pd.Series(downside_target).reindex(base_r.index)
    event = (y < downside_threshold).astype(float)

    df = pd.concat(
        [base_r.rename("base_r"), X, y.rename("target"), event.rename("event")],
        axis=1,
    ).dropna(subset=["base_r"] + list(X.columns))

    p_hat = pd.Series(np.nan, index=df.index, name="p_downside")
    exposure_signal = pd.Series(np.nan, index=df.index, name="exposure_signal")

    last_fit_i = None
    cached_train = None

    for i, ts in enumerate(df.index):
        train_end_i = i - horizon
        if train_end_i < 0:
            continue

        train_df = df.iloc[: train_end_i + 1].dropna(subset=list(X.columns) + ["target", "event"])
        if len(train_df) < min_train:
            continue

        y_train = train_df["event"].astype(int)
        X_train = train_df[X.columns].astype(float)

        if y_train.nunique() < 2:
            p = float(y_train.mean())
        else:
            should_refit = (
                (cached_train is None)
                or (last_fit_i is None)
                or ((i - last_fit_i) >= refit_every)
            )
            if should_refit:
                cached_train = (X_train.copy(), y_train.copy())
                last_fit_i = i

            p = _fit_predict_logit_prob(
                X_train=cached_train[0],
                y_train=cached_train[1],
                x_now=df.loc[ts, X.columns],
            )

        p_hat.loc[ts] = p
        exposure_signal.loc[ts] = prob_to_exposure(p, exposure_map=exposure_map)

    # Signal observed at close t -> exposure applied from next daily return onward.
    exposure_applied = exposure_signal.shift(1).reindex(base_r.index).ffill().fillna(1.0)
    overlay_turnover = exposure_applied.diff().abs().fillna(0.0)
    overlay_cost = (float(overlay_cost_bps) / 1e4) * overlay_turnover

    overlay_r = exposure_applied * base_r - overlay_cost
    overlay_r.name = f"{base_r.name}_overlay" if base_r.name is not None else "overlay_returns"

    diagnostics = pd.concat(
        [
            base_r.rename("base_r"),
            y.rename("target"),
            event.rename("event"),
            p_hat.reindex(base_r.index),
            exposure_signal.reindex(base_r.index),
            exposure_applied.rename("exposure_applied"),
            overlay_turnover.rename("overlay_turnover"),
            overlay_cost.rename("overlay_cost"),
            overlay_r.rename("overlay_r"),
        ],
        axis=1,
    )

    return {
        "overlay_returns": overlay_r,
        "probability": p_hat.reindex(base_r.index),
        "exposure_signal": exposure_signal.reindex(base_r.index),
        "exposure_applied": exposure_applied,
        "diagnostics": diagnostics,
    }


def compare_strategy_metrics(
    strategies: dict[str, pd.Series],
    rf_daily: float | pd.Series | None = 0.0,
    ann: int = 252,
) -> pd.DataFrame:
    """
    Compact comparison table for strategy return series.
    """
    rows = {}
    for name, r in strategies.items():
        rows[name] = perf_metrics(pd.Series(r).dropna(), rf_daily=rf_daily, ann=ann)

    out = pd.DataFrame(rows).T[
        ["n", "CAGR", "CAGR_excess", "ShR", "SoR", "Calmar", "Martin", "MxDD", "Ulcer"]
    ]
    return out


In [ ]:

# ============================================================
# V3 example: IV-aware downside overlay on walk-forward NET PC1
# ============================================================

PC_CORE = "PC1"
H_DOWNSIDE = 21          # maturity-matched monthly horizon (approx. 1 month)
DD_THRESHOLD = -0.1     # event = worst forward return below -5%
MIN_TRAIN_DAYS = 252     # expanding OOS estimation window
REFIT_EVERY = 21         # monthly refit cadence

pc1_base = F_wf_net[PC_CORE].dropna().astype(float).copy()
pc1_base.name = f"{PC_CORE}_wf_net"

iv_X = build_iv_predictor_panel(
    iv_level=iv,
    target_index=pc1_base.index,
    z_window=252,
    z_min_obs=126,
)

y_down = forward_worst_return_from_returns(pc1_base, horizon=H_DOWNSIDE)
df_v3 = pd.concat(
    [
        pc1_base.rename("pc1_base"),
        y_down.rename("worst_fwd_ret"),
        iv_X,
    ],
    axis=1,
).dropna()

df_v3["event_dd"] = (df_v3["worst_fwd_ret"] < DD_THRESHOLD).astype(int)

print("V3 sample shape:", df_v3.shape)
print("Downside event rate:", round(100 * df_v3["event_dd"].mean(), 2), "%")
display(df_v3.head())

print("\nBucket summary by log IV")
display(bucket_summary(df_v3, feature="log_iv", target="worst_fwd_ret", event="event_dd", q=5))

print("\nBucket summary by Δ log IV")
display(bucket_summary(df_v3, feature="dlog_iv", target="worst_fwd_ret", event="event_dd", q=5))

ols_res, ols_tab = hac_ols_summary(
    y=df_v3["worst_fwd_ret"],
    X=df_v3[["log_iv", "dlog_iv"]],
    hac_lags=H_DOWNSIDE,
)
print("\nHAC OLS (worst forward return ~ IV predictors)")
display(ols_tab)

overlay_res = expanding_downside_overlay(
    base_r=pc1_base,
    X=iv_X[["log_iv", "dlog_iv"]],
    downside_target=y_down,
    downside_threshold=DD_THRESHOLD,
    horizon=H_DOWNSIDE,
    min_train=MIN_TRAIN_DAYS,
    refit_every=REFIT_EVERY,
    exposure_map=[
    (0.15, 1.00),
    (0.22, 0.90),
    (0.30, 0.75),
    (np.inf, 0.50),
],
    overlay_cost_bps=0.0,   # optional extension: add explicit overlay trading cost
)

pc1_overlay = overlay_res["overlay_returns"].dropna()
diag_v3 = overlay_res["diagnostics"].dropna(how="all")

rf_v3 = rf_daily_hist.reindex(pc1_base.index).ffill().fillna(0.0)

print(f"\nStrategy comparison (base walk-forward NET PC1 vs IV overlay); RF proxy = {rf_source_used}")
display(
    compare_strategy_metrics(
        {
            "PC1_wf_net": pc1_base,
            "PC1_wf_net_overlay": pc1_overlay,
        },
        rf_daily=rf_v3,
    )
)

eq_v3 = pd.concat(
    [
        (1.0 + pc1_base).cumprod().rename("PC1_wf_net"),
        (1.0 + pc1_overlay).cumprod().rename("PC1_wf_net_overlay"),
    ],
    axis=1,
).dropna()

eq_v3.plot(title="V3 — PC1 walk-forward net vs IV-aware downside overlay")
plt.show()

diag_plot = diag_v3[["p_downside", "exposure_applied"]].dropna()
if not diag_plot.empty:
    diag_plot.plot(
        secondary_y=["exposure_applied"],
        title="V3 — downside-event probability and applied exposure",
    )
    plt.show()

display(diag_v3.tail(10))


In [ ]:
# ============================================================
# V3.1 — Horse race: RV vs downside semivol vs IV
# future RV, forward downside, and OOS downside-event classification
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# -------------------------
# Fallbacks / parameters
# -------------------------
PC_CORE = globals().get("PC_CORE", "PC1")
H = int(globals().get("H_DOWNSIDE", 21))
DD_THR = float(globals().get("DD_THRESHOLD", -0.05))
MIN_TRAIN = int(globals().get("MIN_TRAIN_DAYS", 252))
REFIT = int(globals().get("REFIT_EVERY", 21))
ANN = 252

if "pc1_base" not in globals():
    pc1_base = F_wf_net[PC_CORE].dropna().astype(float).copy()
    pc1_base.name = f"{PC_CORE}_wf_net"

if "iv_X" not in globals():
    iv_X = build_iv_predictor_panel(
        iv_level=iv,
        target_index=pc1_base.index,
        z_window=252,
        z_min_obs=126,
    )

if "y_down" not in globals():
    y_down = forward_worst_return_from_returns(pc1_base, horizon=H)

# -------------------------
# Helpers
# -------------------------
def rolling_realized_vol(r: pd.Series, window: int = 21, ann: int = 252, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    rv = r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)
    return rv.rename(f"rv_{window}")

def rolling_downside_semivol(r: pd.Series, window: int = 21, ann: int = 252, mar: float = 0.0, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    neg = np.minimum(r - mar, 0.0)
    dsv = np.sqrt(ann * pd.Series(neg**2, index=r.index).rolling(window, min_periods=min_obs).mean())
    return dsv.rename(f"dsv_{window}")

def rolling_upside_semivol(r: pd.Series, window: int = 21, ann: int = 252, mar: float = 0.0, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    pos = np.maximum(r - mar, 0.0)
    usv = np.sqrt(ann * pd.Series(pos**2, index=r.index).rolling(window, min_periods=min_obs).mean())
    return usv.rename(f"usv_{window}")

def forward_realized_vol_from_returns(r: pd.Series, horizon: int = 21, ann: int = 252) -> pd.Series:
    """
    Future realized vol over the next H daily returns:
        RV_fwd,t = sqrt(ann) * std(r_{t+1}, ..., r_{t+H})
    """
    r = pd.Series(r).dropna().astype(float)
    x = r.values
    out = np.full(len(x), np.nan)
    for i in range(len(x)):
        j = min(len(x), i + horizon + 1)
        fwd = x[i + 1:j]
        if len(fwd) < max(5, horizon // 2):
            continue
        out[i] = np.std(fwd, ddof=1) * np.sqrt(ann)
    return pd.Series(out, index=r.index, name=f"rv_fwd_{horizon}")

def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    """
    AUC via rank formula (no sklearn dependency).
    """
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc)

def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))

def expanding_prob_forecast(
    X: pd.DataFrame,
    event: pd.Series,
    horizon: int = 21,
    min_train: int = 252,
    refit_every: int = 21,
) -> pd.Series:
    """
    Expanding-window OOS event probability forecast:
    at date t, train only on observations whose forward target is fully known by t
    (non-leaky training end = t - horizon).
    """
    X = pd.DataFrame(X).astype(float)
    event = pd.Series(event).astype(float).reindex(X.index)

    df = pd.concat([X, event.rename("event")], axis=1).dropna()
    p_hat = pd.Series(np.nan, index=df.index, name="p_hat")

    cached_train = None
    last_fit_i = None

    for i, ts in enumerate(df.index):
        train_end_i = i - horizon
        if train_end_i < 0:
            continue

        train_df = df.iloc[: train_end_i + 1].dropna()
        if len(train_df) < min_train:
            continue

        X_train = train_df[X.columns].astype(float)
        y_train = train_df["event"].astype(int)

        if y_train.nunique() < 2:
            p_hat.loc[ts] = float(y_train.mean())
            continue

        should_refit = (
            (cached_train is None)
            or (last_fit_i is None)
            or ((i - last_fit_i) >= refit_every)
        )
        if should_refit:
            cached_train = (X_train.copy(), y_train.copy())
            last_fit_i = i

        p_hat.loc[ts] = _fit_predict_logit_prob(
            X_train=cached_train[0],
            y_train=cached_train[1],
            x_now=df.loc[ts, X.columns],
        )

    return p_hat

def ols_horse_race(df: pd.DataFrame, y_col: str, model_specs: dict, hac_lags: int = 21):
    rows = []
    tabs = {}
    for name, cols in model_specs.items():
        res, tab = hac_ols_summary(
            y=df[y_col],
            X=df[cols],
            hac_lags=hac_lags,
        )
        if res is None or tab.empty:
            continue
        rows.append(
            {
                "model": name,
                "nobs": int(res.nobs),
                "R2": float(getattr(res, "rsquared", np.nan)),
                "AdjR2": float(getattr(res, "rsquared_adj", np.nan)),
                "AIC": float(getattr(res, "aic", np.nan)),
                "BIC": float(getattr(res, "bic", np.nan)),
            }
        )
        tabs[name] = tab
    out = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
    if not out.empty:
        out = out.sort_values(["R2", "AdjR2"], ascending=False)
    return out, tabs

# -------------------------
# Predictors and targets
# -------------------------
rv_21 = rolling_realized_vol(pc1_base, window=H, ann=ANN)
dsv_21 = rolling_downside_semivol(pc1_base, window=H, ann=ANN)
usv_21 = rolling_upside_semivol(pc1_base, window=H, ann=ANN)  # placebo/control
rv_fwd = forward_realized_vol_from_returns(pc1_base, horizon=H, ann=ANN)

df_v31 = pd.concat(
    [
        pc1_base.rename("pc1_base"),
        y_down.rename("worst_fwd_ret"),
        rv_fwd.rename("rv_fwd"),
        rv_21.rename("rv_21"),
        dsv_21.rename("dsv_21"),
        usv_21.rename("usv_21"),
        iv_X[["log_iv", "dlog_iv", "log_iv_z"]],
    ],
    axis=1,
).dropna()

df_v31["event_dd"] = (df_v31["worst_fwd_ret"] < DD_THR).astype(int)
df_v31["iv_over_rv"] = df_v31["log_iv"] - np.log(df_v31["rv_21"].replace(0.0, np.nan))

print("V3.1 sample shape:", df_v31.shape)
print("Event rate:", round(100 * df_v31["event_dd"].mean(), 2), "%")
display(df_v31.head())

# -------------------------
# Exploratory bucket checks
# -------------------------
for feat in ["rv_21", "dsv_21", "usv_21", "log_iv"]:
    print(f"\nBucket summary by {feat}  ->  worst_fwd_ret / event_dd")
    display(bucket_summary(df_v31, feature=feat, target="worst_fwd_ret", event="event_dd", q=5))

# -------------------------
# OLS horse race
# -------------------------
model_specs = {
    "RV_only": ["rv_21"],
    "DSV_only": ["dsv_21"],
    "USV_only": ["usv_21"],
    "IV_only": ["log_iv", "dlog_iv", "log_iv_z"],
    "RV_plus_IV": ["rv_21", "log_iv", "dlog_iv", "log_iv_z"],
    "DSV_plus_IV": ["dsv_21", "log_iv", "dlog_iv", "log_iv_z"],
    "RV_DSV_plus_IV": ["rv_21", "dsv_21", "log_iv", "dlog_iv", "log_iv_z"],
}

print("\nHAC OLS horse race — future realized vol")
ols_rv_tab, ols_rv_detail = ols_horse_race(df_v31, y_col="rv_fwd", model_specs=model_specs, hac_lags=H)
display(ols_rv_tab)

print("\nHAC OLS horse race — worst forward return")
ols_dd_tab, ols_dd_detail = ols_horse_race(df_v31, y_col="worst_fwd_ret", model_specs=model_specs, hac_lags=H)
display(ols_dd_tab)

print("\nDetail: future RV, model = RV_plus_IV")
display(ols_rv_detail.get("RV_plus_IV", pd.DataFrame()))

print("\nDetail: forward downside, model = DSV_plus_IV")
display(ols_dd_detail.get("DSV_plus_IV", pd.DataFrame()))

# -------------------------
# Expanding OOS classification horse race
# -------------------------
clf_specs = {
    "RV_only": ["rv_21"],
    "DSV_only": ["dsv_21"],
    "IV_only": ["log_iv", "dlog_iv", "log_iv_z"],
    "RV_plus_IV": ["rv_21", "log_iv", "dlog_iv", "log_iv_z"],
    "DSV_plus_IV": ["dsv_21", "log_iv", "dlog_iv", "log_iv_z"],
    "ALL": ["rv_21", "dsv_21", "log_iv", "dlog_iv", "log_iv_z"],
}

clf_rows = []
p_store = {}

for name, cols in clf_specs.items():
    p_hat = expanding_prob_forecast(
        X=df_v31[cols],
        event=df_v31["event_dd"],
        horizon=H,
        min_train=MIN_TRAIN,
        refit_every=REFIT,
    )
    p_store[name] = p_hat

    yy = df_v31["event_dd"].reindex(p_hat.index)
    valid = pd.concat([yy.rename("y"), p_hat.rename("p")], axis=1).dropna()

    clf_rows.append(
        {
            "model": name,
            "oos_n": len(valid),
            "event_rate": valid["y"].mean() if len(valid) else np.nan,
            "mean_p": valid["p"].mean() if len(valid) else np.nan,
            "AUC": auc_rank(valid["y"], valid["p"]),
            "Brier": brier_score(valid["y"], valid["p"]),
        }
    )

clf_tab = pd.DataFrame(clf_rows).set_index("model").sort_values(["AUC", "Brier"], ascending=[False, True])

print("\nExpanding OOS logit horse race — downside-event classification")
display(clf_tab)

# -------------------------
# Quick visual: probabilities for the best model by AUC
# -------------------------
if not clf_tab.empty:
    best_model = clf_tab.index[0]
    p_best = p_store[best_model].rename("p_downside_best")
    plot_df = pd.concat([df_v31["event_dd"], p_best], axis=1).dropna()
    print(f"\nBest OOS classifier by AUC: {best_model}")
    display(plot_df.tail(10))

    ax = plot_df["p_downside_best"].plot(title=f"V3.1 — Best OOS downside probability ({best_model})")
    ax.set_ylabel("probability")

In [ ]:
# ============================================================
# V3.2 — Statistical significance tests before any overlay
# 1) Logit: event ~ logIV_{t-1} + dIV^+_t + dIV^-_t
# 2) Future RV ~ lagged RV (Barroso-Santa Clara style) and HAR-RV
# 3) Logit + acceleration: + d2 logIV_t
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# -------------------------
# Parameters / fallbacks
# -------------------------
PC_CORE = globals().get("PC_CORE", "PC1")
H = int(globals().get("H_DOWNSIDE", 21))          # forecast horizon
DD_THR = float(globals().get("DD_THRESHOLD", -0.05))
ANN = 252

# Core return series
if "pc1_base" not in globals():
    pc1_base = F_wf_net[PC_CORE].dropna().astype(float).copy()
    pc1_base.name = f"{PC_CORE}_wf_net"
else:
    pc1_base = pd.Series(pc1_base).dropna().astype(float).copy()

# IV panel
if "iv_X" not in globals():
    iv_X = build_iv_predictor_panel(
        iv_level=iv,
        target_index=pc1_base.index,
        z_window=252,
        z_min_obs=126,
    )

# Downside target (worst forward return) and event
if "y_down" not in globals():
    y_down = forward_worst_return_from_returns(pc1_base, horizon=H)

# -------------------------
# Helpers
# -------------------------
def rolling_realized_vol(r: pd.Series, window: int = 21, ann: int = 252, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    return (r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)).rename(f"rv_{window}")

def forward_realized_vol_from_returns(r: pd.Series, horizon: int = 21, ann: int = 252) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    x = r.values
    out = np.full(len(x), np.nan)
    for i in range(len(x)):
        j = min(len(x), i + horizon + 1)
        fwd = x[i + 1:j]
        if len(fwd) < max(5, horizon // 2):
            continue
        out[i] = np.std(fwd, ddof=1) * np.sqrt(ann)
    return pd.Series(out, index=r.index, name=f"rv_fwd_{horizon}")

def fit_glm_binom_hac(y: pd.Series, X: pd.DataFrame, hac_lags: int = 21):
    """
    Binomial GLM with HAC covariance.
    """
    df = pd.concat([pd.Series(y, name="y"), pd.DataFrame(X)], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return None, pd.DataFrame()
    yv = df["y"].astype(float)
    Xv = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")
    try:
        res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags}
        )
    except Exception:
        # fallback without HAC if needed
        res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit()
    tab = pd.DataFrame({
        "coef": res.params,
        "zstat": res.tvalues,
        "pvalue": res.pvalues,
    })
    try:
        tab["odds_ratio"] = np.exp(res.params)
    except Exception:
        pass
    return res, tab

def fit_ols_hac(y: pd.Series, X: pd.DataFrame, hac_lags: int = 21):
    """
    OLS with HAC covariance.
    """
    df = pd.concat([pd.Series(y, name="y"), pd.DataFrame(X)], axis=1).dropna()
    if df.empty:
        return None, pd.DataFrame()
    yv = df["y"].astype(float)
    Xv = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")
    res = sm.OLS(yv, Xv).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    tab = pd.DataFrame({
        "coef": res.params,
        "tstat_HAC": res.tvalues,
        "pvalue_HAC": res.pvalues,
    })
    return res, tab

def model_summary_row(name: str, res, model_type: str):
    row = {"model": name, "type": model_type}
    if res is None:
        row.update({"nobs": np.nan, "AIC": np.nan, "BIC": np.nan, "pseudo_R2_or_R2": np.nan})
        return row
    row["nobs"] = int(getattr(res, "nobs", np.nan))
    row["AIC"] = float(getattr(res, "aic", np.nan))
    row["BIC"] = float(getattr(res, "bic", np.nan))
    if model_type == "logit":
        try:
            # McFadden pseudo-R2
            row["pseudo_R2_or_R2"] = 1.0 - res.llf / res.llnull
        except Exception:
            row["pseudo_R2_or_R2"] = np.nan
    else:
        row["pseudo_R2_or_R2"] = float(getattr(res, "rsquared", np.nan))
    return row

# -------------------------
# Build clean feature set
# -------------------------
df = pd.concat(
    [
        pc1_base.rename("pc1_base"),
        y_down.rename("worst_fwd_ret"),
        iv_X[["log_iv", "dlog_iv"]],
    ],
    axis=1,
).dropna()

df["event_dd"] = (df["worst_fwd_ret"] < DD_THR).astype(int)

# Clean logistic features:
# logIV_{t-1}, dIV^+_t, dIV^-_t
df["log_iv_lag1"] = df["log_iv"].shift(1)
df["dlog_iv_plus"] = np.maximum(df["dlog_iv"], 0.0)
df["dlog_iv_minus"] = np.minimum(df["dlog_iv"], 0.0)

# Acceleration = second difference of log IV:
# d2 logIV_t = dlog_iv_t - dlog_iv_{t-1}
df["d2log_iv"] = df["dlog_iv"] - df["dlog_iv"].shift(1)

# RV features for future-RV regressions
df["rv_21"] = rolling_realized_vol(df["pc1_base"], window=21, ann=ANN)
df["rv_5"] = rolling_realized_vol(df["pc1_base"], window=5, ann=ANN)
df["rv_fwd_H"] = forward_realized_vol_from_returns(df["pc1_base"], horizon=H, ann=ANN)

# HAR-style lags
df["rv_21_lag1"] = df["rv_21"].shift(1)
df["rv_5_lag1"] = df["rv_5"].shift(1)
df["rv_21_lag5_mean"] = df["rv_21"].shift(1).rolling(5, min_periods=5).mean()
df["rv_21_lag21_mean"] = df["rv_21"].shift(1).rolling(21, min_periods=15).mean()

df_v32 = df.dropna().copy()

print("V3.2 sample shape:", df_v32.shape)
print("Downside event rate:", round(100 * df_v32["event_dd"].mean(), 2), "%")
display(
    df_v32[
        [
            "pc1_base", "worst_fwd_ret", "event_dd",
            "log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus", "d2log_iv",
            "rv_fwd_H", "rv_21_lag1", "rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"
        ]
    ].head()
)

# ============================================================
# 1) Logistic — clean Version 1
# ============================================================
X_logit_v1 = df_v32[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus"]]
res_logit_v1, tab_logit_v1 = fit_glm_binom_hac(df_v32["event_dd"], X_logit_v1, hac_lags=H)

print("\nLogit V1 — event ~ logIV_{t-1} + dIV^+_t + dIV^-_t")
display(tab_logit_v1)

# ============================================================
# 2a) Future RV ~ lagged RV (Barroso–Santa Clara style)
# ============================================================
X_rv_bs = df_v32[["rv_21_lag1"]]
res_rv_bs, tab_rv_bs = fit_ols_hac(df_v32["rv_fwd_H"], X_rv_bs, hac_lags=H)

print("\nFuture RV regression — Barroso/Santa Clara style: RV_fwd ~ RV_{t-1}")
display(tab_rv_bs)

# ============================================================
# 2b) Future RV ~ HAR-RV style (Corsi-like)
#     using short-, weekly-, monthly-type components
# ============================================================
X_rv_har = df_v32[["rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"]]
res_rv_har, tab_rv_har = fit_ols_hac(df_v32["rv_fwd_H"], X_rv_har, hac_lags=H)

print("\nFuture RV regression — HAR style: RV_fwd ~ RV_5 + mean(RV_21, lag 1..5) + mean(RV_21, lag 1..21)")
display(tab_rv_har)

# ============================================================
# 3) Logistic — Version 1 + acceleration
# ============================================================
X_logit_acc = df_v32[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus", "d2log_iv"]]
res_logit_acc, tab_logit_acc = fit_glm_binom_hac(df_v32["event_dd"], X_logit_acc, hac_lags=H)

print("\nLogit V1 + acceleration — event ~ logIV_{t-1} + dIV^+_t + dIV^-_t + d2logIV_t")
display(tab_logit_acc)

# ============================================================
# Compact model comparison
# ============================================================
cmp = pd.DataFrame([
    model_summary_row("Logit_V1", res_logit_v1, "logit"),
    model_summary_row("RV_BS", res_rv_bs, "ols"),
    model_summary_row("RV_HAR", res_rv_har, "ols"),
    model_summary_row("Logit_V1_plus_acc", res_logit_acc, "logit"),
]).set_index("model")

print("\nCompact model comparison")
display(cmp)

# ============================================================
# Quick interpretation helpers
# ============================================================
print("\nSignificance cheat-sheet:")
print("- For logit models, look first at p-values of log_iv_lag1, dlog_iv_plus, dlog_iv_minus, d2log_iv.")
print("- dlog_iv_plus significant with positive coef  => IV up-shocks raise downside-event probability.")
print("- dlog_iv_minus significant with negative coef => IV down-shocks reduce downside-event probability.")
print("- d2log_iv significant                         => acceleration/deceleration of IV adds information.")
print("- For future RV models, significant lagged RV / HAR terms => volatility-targeting overlay becomes a serious candidate.")

In [ ]:
# ============================================================
# V3.3 — Realized volatility vs forward drawdown: bucket tests
# CCA-style nonparametric diagnostic
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

PC_CORE = globals().get("PC_CORE", "PC1")
H = int(globals().get("H_DOWNSIDE", 21))
DD_THR = float(globals().get("DD_THRESHOLD", -0.05))
ANN = 252

if "pc1_base" not in globals():
    pc1_base = F_wf_net[PC_CORE].dropna().astype(float).copy()
    pc1_base.name = f"{PC_CORE}_wf_net"
else:
    pc1_base = pd.Series(pc1_base).dropna().astype(float).copy()

if "y_down" not in globals():
    y_down = forward_worst_return_from_returns(pc1_base, horizon=H)

def rolling_realized_vol(r: pd.Series, window: int = 21, ann: int = 252, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    return (r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)).rename(f"rv_{window}")

def rolling_downside_semivol(r: pd.Series, window: int = 21, ann: int = 252, mar: float = 0.0, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    neg = np.minimum(r - mar, 0.0)
    return np.sqrt(
        ann * pd.Series(neg**2, index=r.index).rolling(window, min_periods=min_obs).mean()
    ).rename(f"dsv_{window}")

def bucket_summary_ext(df: pd.DataFrame, feature: str, target: str, event: str, q: int = 4) -> pd.DataFrame:
    tmp = df[[feature, target, event]].dropna().copy()
    tmp["bucket"] = pd.qcut(tmp[feature], q=q, duplicates="drop")
    out = tmp.groupby("bucket", observed=False).agg(
        n=(feature, "size"),
        feature_mean=(feature, "mean"),
        target_mean=(target, "mean"),
        target_median=(target, "median"),
        target_p10=(target, lambda x: x.quantile(0.10)),
        target_p25=(target, lambda x: x.quantile(0.25)),
        event_rate=(event, "mean"),
    )
    out["event_rate"] = 100 * out["event_rate"]
    return out

# ---------------------------------------
# Build sample
# ---------------------------------------
rv_21 = rolling_realized_vol(pc1_base, window=21, ann=ANN)
rv_63 = rolling_realized_vol(pc1_base, window=63, ann=ANN)
dsv_21 = rolling_downside_semivol(pc1_base, window=21, ann=ANN)
dsv_63 = rolling_downside_semivol(pc1_base, window=63, ann=ANN)

df_v33 = pd.concat(
    [
        pc1_base.rename("pc1_base"),
        y_down.rename("worst_fwd_ret"),
        rv_21.rename("rv_21"),
        rv_63.rename("rv_63"),
        dsv_21.rename("dsv_21"),
        dsv_63.rename("dsv_63"),
    ],
    axis=1,
).dropna()

df_v33["event_dd"] = (df_v33["worst_fwd_ret"] < DD_THR).astype(int)

print("V3.3 sample shape:", df_v33.shape)
print("Event rate:", round(100 * df_v33["event_dd"].mean(), 2), "%")
display(df_v33.head())

# ---------------------------------------
# Quartile bucket summaries
# ---------------------------------------
for feat in ["rv_21", "dsv_21", "rv_63", "dsv_63"]:
    print(f"\nQuartile buckets by {feat} -> forward DD / downside event")
    display(bucket_summary_ext(df_v33, feature=feat, target="worst_fwd_ret", event="event_dd", q=4))

# ---------------------------------------
# Simple monotonicity view
# ---------------------------------------
mono_rows = []
for feat in ["rv_21", "dsv_21", "rv_63", "dsv_63"]:
    tab = bucket_summary_ext(df_v33, feature=feat, target="worst_fwd_ret", event="event_dd", q=4).reset_index(drop=True)
    mono_rows.append({
        "feature": feat,
        "Q1_event_rate_%": tab.loc[0, "event_rate"],
        "Q4_event_rate_%": tab.loc[len(tab)-1, "event_rate"],
        "Q1_p10": tab.loc[0, "target_p10"],
        "Q4_p10": tab.loc[len(tab)-1, "target_p10"],
        "delta_event_rate_pp": tab.loc[len(tab)-1, "event_rate"] - tab.loc[0, "event_rate"],
        "delta_p10": tab.loc[len(tab)-1, "target_p10"] - tab.loc[0, "target_p10"],
    })

mono_tab = pd.DataFrame(mono_rows).set_index("feature")
print("\nMonotonicity check: Q4 vs Q1")
display(mono_tab.sort_values("delta_event_rate_pp", ascending=False))

In [ ]:
# ============================================================
# V3.4 — Proper development / OOS design for the overlay model
# Development sample: up to TRAIN_END_DEFAULT
# OOS sample: from TEST_START_DEFAULT onward
#
# IMPORTANT:
# with START="2015-01-01" and WF_TRAIN_YEARS=5, the earliest
# tradable walk-forward series starts around 2020, not 2015.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# -------------------------
# Core parameters
# -------------------------
PC_CORE = globals().get("PC_CORE", "PC1")
H = int(globals().get("H_DOWNSIDE", 21))
DD_THR = float(globals().get("DD_THRESHOLD", -0.05))
ANN = 252

DEV_END = pd.Timestamp(globals().get("TRAIN_END_DEFAULT", "2022-12-31"))
OOS_START = pd.Timestamp(globals().get("TEST_START_DEFAULT", "2023-01-01"))

# Earliest feasible walk-forward start given current raw data span and warm-up
WF_FIRST_TRADABLE = (pd.Timestamp(globals().get("START", "2015-01-01")) + pd.DateOffset(years=WF_TRAIN_YEARS)).normalize()

print("Earliest feasible walk-forward start with current settings:", WF_FIRST_TRADABLE.date())
print("Development end:", DEV_END.date(), "| OOS start:", OOS_START.date())

# -------------------------
# Helpers
# -------------------------
def rolling_realized_vol(r: pd.Series, window: int = 21, ann: int = 252, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    return (r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)).rename(f"rv_{window}")

def forward_realized_vol_from_returns(r: pd.Series, horizon: int = 21, ann: int = 252) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    x = r.values
    out = np.full(len(x), np.nan)
    for i in range(len(x)):
        j = min(len(x), i + horizon + 1)
        fwd = x[i + 1:j]
        if len(fwd) < max(5, horizon // 2):
            continue
        out[i] = np.std(fwd, ddof=1) * np.sqrt(ann)
    return pd.Series(out, index=r.index, name=f"rv_fwd_{horizon}")

def fit_glm_binom_hac(y: pd.Series, X: pd.DataFrame, hac_lags: int = 21):
    df = pd.concat([pd.Series(y, name="y"), pd.DataFrame(X)], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return None, pd.DataFrame()

    yv = df["y"].astype(float)
    Xv = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")

    try:
        res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags}
        )
    except Exception:
        res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit()

    tab = pd.DataFrame({
        "coef": res.params,
        "zstat": res.tvalues,
        "pvalue": res.pvalues,
        "odds_ratio": np.exp(res.params),
    })
    return res, tab

def fit_ols_hac(y: pd.Series, X: pd.DataFrame, hac_lags: int = 21):
    df = pd.concat([pd.Series(y, name="y"), pd.DataFrame(X)], axis=1).dropna()
    if df.empty:
        return None, pd.DataFrame()

    yv = df["y"].astype(float)
    Xv = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")
    res = sm.OLS(yv, Xv).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})

    tab = pd.DataFrame({
        "coef": res.params,
        "tstat_HAC": res.tvalues,
        "pvalue_HAC": res.pvalues,
    })
    return res, tab

def model_summary_row(name: str, res, model_type: str):
    row = {"model": name, "type": model_type}
    if res is None:
        row.update({"nobs": np.nan, "AIC": np.nan, "BIC": np.nan, "pseudo_R2_or_R2": np.nan})
        return row
    row["nobs"] = int(getattr(res, "nobs", np.nan))
    row["AIC"] = float(getattr(res, "aic", np.nan))
    row["BIC"] = float(getattr(res, "bic", np.nan))
    if model_type == "logit":
        try:
            row["pseudo_R2_or_R2"] = 1.0 - res.llf / res.llnull
        except Exception:
            row["pseudo_R2_or_R2"] = np.nan
    else:
        row["pseudo_R2_or_R2"] = float(getattr(res, "rsquared", np.nan))
    return row

def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))

def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))

# -------------------------
# 1) Build a longer walk-forward tradable series
# -------------------------
# This is the series on which the overlay model should be developed/tested.
F_wf_long, F_wf_net_long, W_hist_long = walk_forward_pca(
    R=R,
    start_test=str(WF_FIRST_TRADABLE.date()),
    train_years=WF_TRAIN_YEARS,
    step_months=WF_STEP_MONTHS,
    k=WF_K_CHOICE,
    gross_norm=GROSS_NORM,
    tc_bps=None,
    asset_costs=asset_costs,
    charge_initial_rebalance=False,
)

pc1_long = F_wf_net_long[PC_CORE].dropna().astype(float).copy()
pc1_long.name = f"{PC_CORE}_wf_net_long"

print("Long walk-forward net series:")
print(pc1_long.index.min().date(), "->", pc1_long.index.max().date(), "| n =", len(pc1_long))

# -------------------------
# 2) Build full panel on the long tradable series
# -------------------------
iv_panel_long = build_iv_predictor_panel(
    iv_level=iv,
    target_index=pc1_long.index,
    z_window=252,
    z_min_obs=126,
)

y_down_long = forward_worst_return_from_returns(pc1_long, horizon=H)

panel = pd.concat(
    [
        pc1_long.rename("pc1_base"),
        y_down_long.rename("worst_fwd_ret"),
        iv_panel_long[["log_iv", "dlog_iv"]],
    ],
    axis=1,
).dropna()

panel["event_dd"] = (panel["worst_fwd_ret"] < DD_THR).astype(int)

# Logistic V1 features
panel["log_iv_lag1"] = panel["log_iv"].shift(1)
panel["dlog_iv_plus"] = np.maximum(panel["dlog_iv"], 0.0)
panel["dlog_iv_minus"] = np.minimum(panel["dlog_iv"], 0.0)

# Acceleration
panel["d2log_iv"] = panel["dlog_iv"] - panel["dlog_iv"].shift(1)

# RV features
panel["rv_21"] = rolling_realized_vol(panel["pc1_base"], window=21, ann=ANN)
panel["rv_5"] = rolling_realized_vol(panel["pc1_base"], window=5, ann=ANN)
panel["rv_fwd_H"] = forward_realized_vol_from_returns(panel["pc1_base"], horizon=H, ann=ANN)

# HAR-style lagged components
panel["rv_21_lag1"] = panel["rv_21"].shift(1)
panel["rv_5_lag1"] = panel["rv_5"].shift(1)
panel["rv_21_lag5_mean"] = panel["rv_21"].shift(1).rolling(5, min_periods=5).mean()
panel["rv_21_lag21_mean"] = panel["rv_21"].shift(1).rolling(21, min_periods=15).mean()

panel = panel.dropna().copy()

# -------------------------
# 3) Proper split: development vs OOS
# -------------------------
dev = panel.loc[:DEV_END].copy()
oos = panel.loc[OOS_START:].copy()

print("\nPanel ranges after lag/rolling/forward construction:")
print("Full panel :", panel.index.min().date(), "->", panel.index.max().date(), "| n =", len(panel))
print("DEV sample :", dev.index.min().date() if len(dev) else None, "->", dev.index.max().date() if len(dev) else None, "| n =", len(dev))
print("OOS sample :", oos.index.min().date() if len(oos) else None, "->", oos.index.max().date() if len(oos) else None, "| n =", len(oos))
print("DEV downside event rate:", round(100 * dev["event_dd"].mean(), 2) if len(dev) else np.nan, "%")
print("OOS downside event rate:", round(100 * oos["event_dd"].mean(), 2) if len(oos) else np.nan, "%")

display(dev.head())

# -------------------------
# 4) DEVELOPMENT-SAMPLE significance tests
# -------------------------
# 4a) Logit V1
X_logit_v1_dev = dev[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus"]]
res_logit_v1_dev, tab_logit_v1_dev = fit_glm_binom_hac(dev["event_dd"], X_logit_v1_dev, hac_lags=H)

print("\nDEV — Logit V1: event ~ logIV_{t-1} + dIV^+_t + dIV^-_t")
display(tab_logit_v1_dev)

# 4b) Logit V1 + acceleration
X_logit_acc_dev = dev[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus", "d2log_iv"]]
res_logit_acc_dev, tab_logit_acc_dev = fit_glm_binom_hac(dev["event_dd"], X_logit_acc_dev, hac_lags=H)

print("\nDEV — Logit V1 + acceleration")
display(tab_logit_acc_dev)

# 4c) Future RV ~ lagged RV (Barroso–Santa Clara style)
X_rv_bs_dev = dev[["rv_21_lag1"]]
res_rv_bs_dev, tab_rv_bs_dev = fit_ols_hac(dev["rv_fwd_H"], X_rv_bs_dev, hac_lags=H)

print("\nDEV — Future RV regression: RV_fwd ~ RV_{t-1}")
display(tab_rv_bs_dev)

# 4d) Future RV ~ HAR-RV
X_rv_har_dev = dev[["rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"]]
res_rv_har_dev, tab_rv_har_dev = fit_ols_hac(dev["rv_fwd_H"], X_rv_har_dev, hac_lags=H)

print("\nDEV — Future RV regression: HAR style")
display(tab_rv_har_dev)

# -------------------------
# 5) Compact comparison on DEVELOPMENT sample
# -------------------------
cmp_dev = pd.DataFrame([
    model_summary_row("DEV_Logit_V1", res_logit_v1_dev, "logit"),
    model_summary_row("DEV_Logit_V1_plus_acc", res_logit_acc_dev, "logit"),
    model_summary_row("DEV_RV_BS", res_rv_bs_dev, "ols"),
    model_summary_row("DEV_RV_HAR", res_rv_har_dev, "ols"),
]).set_index("model")

print("\nDevelopment-sample model comparison")
display(cmp_dev)

# -------------------------
# 6) Quick frozen-coefficient OOS diagnostic (no overlay yet)
# -------------------------
oos_diag_rows = []

if len(oos) > 0 and res_logit_v1_dev is not None:
    X_oos_v1 = sm.add_constant(oos[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus"]], has_constant="add")
    p_oos_v1 = pd.Series(res_logit_v1_dev.predict(X_oos_v1), index=oos.index, name="p_oos_v1")
    oos_diag_rows.append({
        "model": "Frozen_Logit_V1",
        "oos_n": len(p_oos_v1.dropna()),
        "AUC": auc_rank(oos["event_dd"], p_oos_v1),
        "Brier": brier_score(oos["event_dd"], p_oos_v1),
        "mean_p": float(p_oos_v1.mean()),
        "event_rate": float(oos["event_dd"].mean()),
    })

if len(oos) > 0 and res_logit_acc_dev is not None:
    X_oos_acc = sm.add_constant(oos[["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus", "d2log_iv"]], has_constant="add")
    p_oos_acc = pd.Series(res_logit_acc_dev.predict(X_oos_acc), index=oos.index, name="p_oos_acc")
    oos_diag_rows.append({
        "model": "Frozen_Logit_V1_plus_acc",
        "oos_n": len(p_oos_acc.dropna()),
        "AUC": auc_rank(oos["event_dd"], p_oos_acc),
        "Brier": brier_score(oos["event_dd"], p_oos_acc),
        "mean_p": float(p_oos_acc.mean()),
        "event_rate": float(oos["event_dd"].mean()),
    })

if oos_diag_rows:
    oos_diag = pd.DataFrame(oos_diag_rows).set_index("model")
    print("\nQuick frozen-coefficient OOS diagnostic (no overlay yet)")
    display(oos_diag)

print("\nWhat to look at now:")
print("1) Development sample first: significance and signs.")
print("2) Only if a model is sensible in DEV do we care about frozen OOS diagnostics.")
print("3) If RV/HAR is the stronger block again, the next overlay to build should be a volatility overlay, not the IV-logit one.")

In [ ]:
# ============================================================
# V5 — HAR-RV + residual bootstrap predictive distribution
# Goal: estimate P( RV_fwd_H > tau | information at t )
# using HAR mean dynamics + bootstrapped residual uncertainty
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# ------------------------------------------------------------
# Preconditions:
# expects dev / oos from V3.4 already available
# with columns:
#   rv_fwd_H, rv_5_lag1, rv_21_lag5_mean, rv_21_lag21_mean
# ------------------------------------------------------------
assert "dev" in globals() and "oos" in globals(), "Run V3.4 first so that dev and oos exist."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BOOT_B = 5000                 # number of bootstrap draws
TAU_MODE = "dev_percentile"   # "dev_percentile" or "absolute"
TAU_PERCENTILE = 0.80         # e.g. 0.80, 0.85, 0.90
TAU_ABS = 0.30                # used only if TAU_MODE == "absolute"

HAR_COLS = ["rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"]
Y_COL = "rv_fwd_H"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc)

def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))

def bucket_prob_calibration(y_true: pd.Series, p_hat: pd.Series, q: int = 5) -> pd.DataFrame:
    tmp = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna().copy()
    if tmp.empty:
        return pd.DataFrame()
    tmp["bucket"] = pd.qcut(tmp["p"], q=q, duplicates="drop")
    out = tmp.groupby("bucket", observed=False).agg(
        n=("y", "size"),
        mean_p=("p", "mean"),
        realized_rate=("y", "mean"),
    )
    out["realized_rate"] *= 100
    out["mean_p"] *= 100
    return out

# ------------------------------------------------------------
# 1) Fit HAR on development sample
# ------------------------------------------------------------
har_dev = dev[[Y_COL] + HAR_COLS].dropna().copy()
X_dev = sm.add_constant(har_dev[HAR_COLS], has_constant="add")
y_dev = har_dev[Y_COL].astype(float)

har_res = sm.OLS(y_dev, X_dev).fit()
har_resid = (y_dev - har_res.fittedvalues).dropna()

print("HAR fit on development sample")
print("n_dev:", len(har_dev))
display(pd.DataFrame({
    "coef": har_res.params,
    "tstat": har_res.tvalues,
    "pvalue": har_res.pvalues,
}))

# ------------------------------------------------------------
# 2) Choose threshold tau
# ------------------------------------------------------------
if TAU_MODE == "dev_percentile":
    tau = float(dev[Y_COL].dropna().quantile(TAU_PERCENTILE))
    tau_label = f"dev p{int(TAU_PERCENTILE*100)}"
else:
    tau = float(TAU_ABS)
    tau_label = f"absolute {TAU_ABS:.4f}"

print(f"\nChosen volatility threshold tau = {tau:.6f} ({tau_label})")

# ------------------------------------------------------------
# 3) OOS predictive distribution via residual bootstrap
#    Frozen-coefficient version first (clean baseline)
# ------------------------------------------------------------
har_oos = oos[[Y_COL] + HAR_COLS].dropna().copy()
X_oos = sm.add_constant(har_oos[HAR_COLS], has_constant="add")
mu_oos = pd.Series(har_res.predict(X_oos), index=har_oos.index, name="mu_hat")

# bootstrap residual draws
resid_pool = har_resid.values.astype(float)
boot_draws = np.random.choice(resid_pool, size=(len(mu_oos), BOOT_B), replace=True)

# predictive draws: mean + resampled residual
pred_draws = mu_oos.values.reshape(-1, 1) + boot_draws

# optional clipping to nonnegative values
pred_draws = np.clip(pred_draws, 1e-8, None)

# predictive summaries
p_exceed = pd.Series((pred_draws > tau).mean(axis=1), index=mu_oos.index, name="p_rv_exceed")
q90 = pd.Series(np.quantile(pred_draws, 0.90, axis=1), index=mu_oos.index, name="rv_q90")
pred_mean = pd.Series(pred_draws.mean(axis=1), index=mu_oos.index, name="rv_pred_mean")
pred_std = pd.Series(pred_draws.std(axis=1, ddof=1), index=mu_oos.index, name="rv_pred_std")

# realized event
event_oos = (har_oos[Y_COL] > tau).astype(int).rename("event_rv_exceed")

# ------------------------------------------------------------
# 4) Diagnostics
# ------------------------------------------------------------
diag = pd.DataFrame({
    "realized_rv_fwd": har_oos[Y_COL],
    "rv_pred_mean": pred_mean,
    "rv_pred_std": pred_std,
    "rv_q90": q90,
    "p_rv_exceed": p_exceed,
    "event_rv_exceed": event_oos,
})

print("\nOOS predictive distribution diagnostics")
summary_diag = pd.DataFrame([{
    "oos_n": len(diag),
    "tau": tau,
    "event_rate_%": 100 * diag["event_rv_exceed"].mean(),
    "mean_pred_p_%": 100 * diag["p_rv_exceed"].mean(),
    "AUC": auc_rank(diag["event_rv_exceed"], diag["p_rv_exceed"]),
    "Brier": brier_score(diag["event_rv_exceed"], diag["p_rv_exceed"]),
}]).T
summary_diag.columns = ["value"]
display(summary_diag)

print("\nProbability calibration by buckets")
display(bucket_prob_calibration(diag["event_rv_exceed"], diag["p_rv_exceed"], q=5))

print("\nTail of OOS predictive table")
display(diag.tail(10))

# ------------------------------------------------------------
# 5) Simple plots
# ------------------------------------------------------------
ax = diag[["realized_rv_fwd", "rv_pred_mean", "rv_q90"]].plot(
    title=f"V5 — OOS HAR predictive distribution | tau={tau:.3f}"
)
ax.set_ylabel("forward realized volatility")

ax2 = diag["p_rv_exceed"].plot(
    title=f"V5 — OOS probability that forward RV exceeds tau={tau:.3f}"
)
ax2.set_ylabel("probability")

# ------------------------------------------------------------
# 6) Save useful objects for next overlay step
# ------------------------------------------------------------
har_boot_oos = diag.copy()
har_boot_oos["tau"] = tau
har_boot_oos["tau_mode"] = tau_label

print("\nSaved object: har_boot_oos")

In [ ]:
# ============================================================
# V5.1 — Slow overlay from HAR-bootstrap volatility score
# Weekly rebalance + hysteresis + mild de-risking
# Uses har_boot_oos from V5 and oos from V3.4
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

assert "har_boot_oos" in globals(), "Run V5 first so that har_boot_oos exists."
assert "oos" in globals(), "Run V3.4 first so that oos exists."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BASE_COL = "pc1_base"
SCORE_COL = "p_rv_exceed"

# exposure states
EXP_NORMAL = 1.00
EXP_DEF = 0.75
EXP_EXT = 0.50

# hysteresis on expanding percentiles of the score
ENTER_DEF_Q = 0.90
EXIT_DEF_Q  = 0.75
ENTER_EXT_Q = 0.98
EXIT_EXT_Q  = 0.90

MIN_HIST_WEEKS = 26          # before this, stay fully invested
OVERLAY_COST_BPS = 0.0       # can raise later if you want

# ------------------------------------------------------------
# Helper: performance table
# ------------------------------------------------------------
def perf_table(return_dict: dict[str, pd.Series], ann: int = 252) -> pd.DataFrame:
    rows = []
    for name, r in return_dict.items():
        r = pd.Series(r).dropna().astype(float)
        if r.empty:
            continue

        wealth = (1.0 + r).cumprod()
        n = len(r)
        years = n / ann

        cagr = wealth.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
        vol = r.std(ddof=1) * np.sqrt(ann)
        mu = r.mean() * ann
        sharpe = mu / vol if vol > 0 else np.nan

        downside = r[r < 0]
        dvol = downside.std(ddof=1) * np.sqrt(ann) if len(downside) > 1 else np.nan
        sortino = mu / dvol if pd.notna(dvol) and dvol > 0 else np.nan

        peak = wealth.cummax()
        dd = wealth / peak - 1.0
        max_dd = dd.min()
        calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

        ulcer = np.sqrt(np.mean((100 * dd) ** 2)) / 100
        martin = cagr / ulcer if ulcer > 0 else np.nan

        rows.append({
            "n": n,
            "CAGR": cagr,
            "AnnRet": mu,
            "AnnVol": vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MaxDD": max_dd,
            "Calmar": calmar,
            "Ulcer": ulcer,
            "Martin": martin,
        })

    return pd.DataFrame(rows, index=list(return_dict.keys()))

# ------------------------------------------------------------
# 1) Build daily base series and score series
# ------------------------------------------------------------
base_r = oos[BASE_COL].dropna().astype(float).copy()
score_daily = har_boot_oos[SCORE_COL].dropna().astype(float).copy()

# align on common daily index
common_idx = base_r.index.intersection(score_daily.index)
base_r = base_r.loc[common_idx]
score_daily = score_daily.loc[common_idx]

# weekly rebalance score = last available score each week
score_weekly = score_daily.resample("W-FRI").last().dropna()

# ------------------------------------------------------------
# 2) State machine with expanding percentile thresholds
# ------------------------------------------------------------
state_rows = []
state = "normal"

for i, dt in enumerate(score_weekly.index):
    p_now = float(score_weekly.loc[dt])

    hist = score_weekly.iloc[:i]
    if len(hist) < MIN_HIST_WEEKS:
        q_def_enter = q_def_exit = q_ext_enter = q_ext_exit = np.nan
        state = "normal"
        exposure = EXP_NORMAL
    else:
        q_def_enter = float(hist.quantile(ENTER_DEF_Q))
        q_def_exit  = float(hist.quantile(EXIT_DEF_Q))
        q_ext_enter = float(hist.quantile(ENTER_EXT_Q))
        q_ext_exit  = float(hist.quantile(EXIT_EXT_Q))

        if state == "normal":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now >= q_def_enter:
                state = "defensive"

        elif state == "defensive":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now <= q_def_exit:
                state = "normal"

        elif state == "extreme":
            if p_now <= q_ext_exit:
                # de-escalate with hysteresis
                if p_now <= q_def_exit:
                    state = "normal"
                else:
                    state = "defensive"

        exposure = {
            "normal": EXP_NORMAL,
            "defensive": EXP_DEF,
            "extreme": EXP_EXT,
        }[state]

    state_rows.append({
        "Date": dt,
        "p_score": p_now,
        "q_def_enter": q_def_enter,
        "q_def_exit": q_def_exit,
        "q_ext_enter": q_ext_enter,
        "q_ext_exit": q_ext_exit,
        "state": state,
        "exposure_signal": exposure,
    })

weekly_overlay = pd.DataFrame(state_rows).set_index("Date")

print("Weekly overlay state table")
display(weekly_overlay.tail(15))

# ------------------------------------------------------------
# 3) Push weekly signal to daily frequency
# ------------------------------------------------------------
exposure_signal_daily = weekly_overlay["exposure_signal"].reindex(base_r.index, method="ffill").fillna(EXP_NORMAL)

# apply from next trading day to avoid same-day look-ahead
exposure_applied = exposure_signal_daily.shift(1).fillna(EXP_NORMAL).rename("exposure_applied")

# turnover + optional cost
overlay_turnover = exposure_applied.diff().abs().fillna(0.0).rename("overlay_turnover")
overlay_cost = (OVERLAY_COST_BPS / 10000.0) * overlay_turnover
overlay_r = (exposure_applied * base_r - overlay_cost).rename("overlay_r")

# ------------------------------------------------------------
# 4) Collect diagnostic panel
# ------------------------------------------------------------
overlay_panel = pd.concat(
    [
        base_r.rename("base_r"),
        score_daily.rename("p_score_daily"),
        exposure_signal_daily.rename("exposure_signal_daily"),
        exposure_applied,
        overlay_turnover,
        overlay_cost.rename("overlay_cost"),
        overlay_r,
    ],
    axis=1,
)

print("\nOverlay daily panel tail")
display(overlay_panel.tail(15))

# ------------------------------------------------------------
# 5) Compare performance
# ------------------------------------------------------------
perf = perf_table({
    "PC1_base_OOS": base_r,
    "PC1_slow_overlay": overlay_r,
})

print("\nPerformance comparison")
display(perf)

# ------------------------------------------------------------
# 6) Event concentration check
#    Do high-score / de-risk states line up with realized RV exceedance events?
# ------------------------------------------------------------
if "event_rv_exceed" in har_boot_oos.columns:
    event_daily = har_boot_oos["event_rv_exceed"].reindex(base_r.index)
    event_check = pd.concat(
        [
            event_daily.rename("event_rv_exceed"),
            score_daily.rename("p_score"),
            exposure_signal_daily.rename("exposure_signal"),
        ],
        axis=1,
    ).dropna()

    state_summary = event_check.groupby("exposure_signal", observed=False).agg(
        n=("event_rv_exceed", "size"),
        mean_p=("p_score", "mean"),
        event_rate=("event_rv_exceed", "mean"),
    )
    state_summary["event_rate"] *= 100

    print("\nEvent concentration by exposure state")
    display(state_summary)

# ------------------------------------------------------------
# 7) Save useful objects
# ------------------------------------------------------------
slow_overlay_oos = overlay_panel.copy()
slow_overlay_weekly = weekly_overlay.copy()

print("\nSaved objects: slow_overlay_oos, slow_overlay_weekly")

## 7) Regime-conditioned core-satellite allocation

This section implements the next extension after the naive HIGH $\rightarrow$ RF tests.

**Idea:** keep PC1 as the equity core, but allocate progressively to a defensive satellite only inside the adverse IV regime. The stress intensity is deliberately simple at this stage: an empirical, non-parametric percentile of the IV score **conditional on historical HIGH observations**.

Allocation rule:

$$
R_t^{strategy} = (1-SI_{t-1})R_t^{PC1} + SI_{t-1}R_t^{SAT} - TC_t
$$

where `SI=0` outside HIGH, and within HIGH grows with the current IV score's position inside the historical HIGH-state IV distribution. The satellite is selected from broad, external asset-class proxies using validation-sample stressed correlation and conditional beta versus PC1.

In [ ]:
# ============================================================
# Regime-conditioned core-satellite allocation parameters
# ============================================================

# The split is resolved automatically from the walk-forward/test index unless explicit
# dates below are supplied. This prevents empty validation windows when
# TRAIN_END_DEFAULT / TEST_START_DEFAULT are changed.
SAT_VALIDATION_START = None   # e.g. "2023-01-01"; None => auto
SAT_VALIDATION_END   = None   # e.g. "2024-06-30"; None => auto
SAT_FINAL_OOS_START  = None   # e.g. "2024-07-01"; None => auto

SAT_VALIDATION_FRAC = 0.45
SAT_MIN_VALIDATION_DAYS = 126
SAT_MIN_FINAL_DAYS = 126
SAT_MIN_HIGH_OBS = 30

# Stress-intensity mapping inside HIGH.
# SI_t is the empirical percentile of IV score inside the historical HIGH distribution.
SI_CONDITIONAL_MIN_OBS = 50
SI_GAMMA_GRID = [1.0, 1.5, 2.0]

# Nonlinearity currently tested in the OUTER allocation between Core (PC1) and the tangency satellite.
# The linear inner Tangency -> RF family is ALSO kept below as a separate candidate family.
# g_gamma(SI) = SI ** gamma_core
# Portfolio_t = (1 - g_gamma(SI_t)) * Core + g_gamma(SI_t) * TangencySatellite
CORE_SAT_GAMMA_GRID = [
    0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
    1.00,
    1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 1.40, 1.45, 1.50,
    1.75, 2.00, 2.50, 3.00, 4.00,
]
CORE_SAT_VALIDATION_TOP_K = 3

# Nonlinearity for inner Tangency -> RF motion inside the satellite.
RF_INNER_GAMMA_GRID = [
    0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
    1.00,
    1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 1.40, 1.45, 1.50,
    1.75, 2.00, 2.50, 3.00, 4.00,
]

# External asset-class universe. We keep cash-like sleeves in the downloaded pool
# for benchmarks/diagnostics, but the "risky satellite" screening and tangency
# construction exclude them by default.
SATELLITE_ASSET_CLASS_TICKERS = {
    "Cash_TBill": "BIL",
    "Short_Treasury": "SHY",
    "Intermediate_Treasury": "IEF",
    "Long_Treasury": "TLT",
    "Extended_Duration": "EDV",
    "TIPS": "TIP",
    "Agency_MBS": "MBB",
    "IG_Credit": "LQD",
    "High_Yield_Credit": "HYG",
    "Gold": "GLD",
    "Broad_Commodities": "DBC",
    "US_Dollar": "UUP",
    "US_REITs": "VNQ",
    "DM_ex_US_Equity": "VEA",
    "Emerging_Market_Equity": "EEM",
    "LowVol_US_Equity": "SPLV",
    "Consumer_Staples": "XLP",
    "Utilities": "XLU",
}

SAT_CASHLIKE_COLS = ["Cash_TBill", "Short_Treasury"]

# Data-quality filter for external ETFs after alignment to the test/walk-forward index.
SAT_MIN_COVERAGE = 0.90

# -------------------------
# SCREENING (portfolio-aware)
# -------------------------
# Screening is done on the whole portfolio [PC1 core + candidate asset classes]
# in validation-HIGH only, using conditional risk parity and a common conditional-vol target.
SAT_SCREEN_USE_CASHLIKE = False         # exclude cash-like sleeves from risky-satellite screening
SAT_SCREEN_TARGET_VOL = None            # None => use full-universe RP conditional vol in validation-HIGH
SAT_SCREEN_MAX_WEIGHT = 0.45            # cap in RP screening
SAT_SCREEN_COV_SHRINK = 0.20            # diagonal shrinkage for conditional covariance
SAT_SCREEN_TOP_K = 5                    # max number of selected asset classes for satellite construction
SAT_SCREEN_REQUIRE_POSITIVE_EXCESS = True
SAT_SCREEN_REQUIRE_NONNEG_CAGR = False  # if True, require non-negative CAGR contribution as well

# -------------------------
# SATELLITE CONSTRUCTION
# -------------------------
# After screening, build the actual risky satellite only on the selected asset classes.
SAT_MAXSHARPE_MAX_WEIGHT = 0.45
SAT_MAXSHARPE_COV_SHRINK = 0.20
SAT_MAXSHARPE_ALLOW_CASHLIKE = False    # risky satellite excludes cash-like sleeves

# Outer allocation motion:
# The SAME stress indicator SI_t still defines the adverse-state weight,
# but the mapping from SI_t to the total satellite weight is now nonlinear:
#   g_gamma(SI_t) = SI_t ** gamma_core
#   Total_t = (1 - g_gamma(SI_t)) * Core + g_gamma(SI_t) * TangencySatellite
#
# We are NOT using the Tangency->RF inner motion in this version.

# Final overlay selection across gamma / destination variants.
SAT_VALIDATION_SELECTION_METRIC = "Martin"   # "Martin" or "ShR"

SAT_OVERLAY_TC_BPS = TC_BPS         # incremental allocation-turnover cost between core and satellite
SAT_BOOT_N = 1000
SAT_BOOT_AVG_BLOCK = L_pw if "L_pw" in globals() else 43


In [ ]:
# ============================================================
# Helpers for stress intensity, screening, satellite selection, and strategy evaluation
# ============================================================

def resolve_core_satellite_splits(
    idx: pd.DatetimeIndex,
    validation_start=None,
    validation_end=None,
    final_oos_start=None,
    validation_frac: float = 0.45,
    min_validation_days: int = 126,
    min_final_days: int = 126,
):
    """
    Resolve validation/frozen-OOS dates from the available tradable index.
    If explicit dates are inconsistent or empty, fall back to an automatic split.
    """
    idx = pd.DatetimeIndex(pd.Index(idx).dropna()).sort_values().unique()
    if len(idx) < min_validation_days + min_final_days:
        raise ValueError(
            f"Not enough observations for validation/final split: n={len(idx)}, "
            f"need at least {min_validation_days + min_final_days}."
        )

    def _to_ts(x):
        return None if x is None else pd.Timestamp(x)

    vs = _to_ts(validation_start)
    ve = _to_ts(validation_end)
    fs = _to_ts(final_oos_start)

    explicit_ok = (vs is not None and ve is not None and fs is not None and vs <= ve < fs)
    if explicit_ok:
        val_idx = idx[(idx >= vs) & (idx <= ve)]
        final_idx = idx[idx >= fs]
        if len(val_idx) >= min_validation_days and len(final_idx) >= min_final_days:
            return val_idx[0], val_idx[-1], final_idx[0], "explicit"

    n = len(idx)
    split_pos = int(np.floor(n * validation_frac))
    split_pos = max(min_validation_days, split_pos)
    split_pos = min(n - min_final_days, split_pos)
    if split_pos <= 0 or split_pos >= n:
        raise ValueError("Could not create a valid automatic validation/final split.")

    val_idx = idx[:split_pos]
    final_idx = idx[split_pos:]
    return val_idx[0], val_idx[-1], final_idx[0], "auto"


def expanding_conditional_percentile(
    x: pd.Series,
    condition: pd.Series,
    min_obs: int = 50,
) -> pd.Series:
    """
    Live-implementable empirical CDF of x_t conditional on past observations
    satisfying condition=True. Percentile at t uses only dates < t.
    """
    x = pd.Series(x).dropna().astype(float).sort_index()
    condition = pd.Series(condition).reindex(x.index).fillna(False).astype(bool)
    out = pd.Series(np.nan, index=x.index, name="conditional_percentile")
    hist_vals = []

    for dt, val in x.items():
        if condition.loc[dt] and len(hist_vals) >= min_obs:
            hv = np.asarray(hist_vals, dtype=float)
            out.loc[dt] = float(np.mean(hv <= val))
        if condition.loc[dt] and np.isfinite(val):
            hist_vals.append(float(val))

    return out.clip(0.0, 1.0)


def build_high_regime_si(
    iv_score: pd.Series,
    regime: pd.Series,
    high_label: str = "HIGH",
    gamma: float = 1.0,
    min_obs: int = 50,
) -> pd.Series:
    """Stress intensity: 0 outside HIGH; inside HIGH, powered conditional IV percentile."""
    common_idx = pd.Index(iv_score.dropna().index).intersection(regime.dropna().index)
    z_ = iv_score.reindex(common_idx).astype(float)
    r_ = regime.reindex(common_idx)
    high = r_.eq(high_label)
    pct = expanding_conditional_percentile(z_, high, min_obs=min_obs)
    si = pct.pow(float(gamma)).where(high, 0.0).fillna(0.0).clip(0.0, 1.0)
    si.name = f"SI_high_iv_pct_gamma_{gamma:g}"
    return si


def _adj_close_from_yf_download(raw, tickers: list[str]) -> pd.DataFrame:
    """Robustly extract adjusted close / close from yfinance output."""
    if raw is None or len(raw) == 0:
        return pd.DataFrame()
    if isinstance(raw.columns, pd.MultiIndex):
        level0 = list(raw.columns.get_level_values(0).unique())
        field = "Adj Close" if "Adj Close" in level0 else "Close"
        out = raw[field].copy()
    else:
        field = "Adj Close" if "Adj Close" in raw.columns else "Close"
        out = raw[[field]].copy()
        out.columns = tickers[:1]
    return out.dropna(how="all")


def download_asset_class_returns(asset_map: dict[str, str], start: str, end: str) -> pd.DataFrame:
    tickers = list(dict.fromkeys(asset_map.values()))
    raw = download(tickers, start=start, end=end, auto_adjust=False, progress=False)
    if raw is None or len(raw) == 0:
        return pd.DataFrame()
    raw = raw.dropna(how="all")
    px_sat = _adj_close_from_yf_download(raw, tickers=tickers)
    inverse = {v: k for k, v in asset_map.items()}
    px_sat = px_sat.rename(columns=inverse)
    cols = [c for c in asset_map.keys() if c in px_sat.columns]
    px_sat = px_sat[cols].dropna(how="all")
    sat_ret = linear_returns_from_prices(px_sat).dropna(how="all")
    sat_ret.index = pd.to_datetime(sat_ret.index)
    return sat_ret


def conditional_beta_corr_table(
    core: pd.Series,
    candidates: pd.DataFrame,
    mask: pd.Series,
    rf_daily: pd.Series | float | None = None,
    ann: int = 252,
    min_obs: int = 30,
) -> pd.DataFrame:
    """Diagnostics of satellite candidates in the stressed/HIGH subset."""
    core = pd.Series(core).rename("PC1_core").astype(float)
    candidates = pd.DataFrame(candidates).astype(float)
    mask = pd.Series(mask).reindex(core.index).fillna(False).astype(bool)

    rows = []
    for c in candidates.columns:
        tmp = pd.concat([core, candidates[c].rename(c)], axis=1).dropna()
        tmp = tmp.loc[mask.reindex(tmp.index).fillna(False)]
        if len(tmp) < min_obs or tmp["PC1_core"].var(ddof=1) <= 0:
            continue
        cov = float(np.cov(tmp[c], tmp["PC1_core"], ddof=1)[0, 1])
        beta = cov / float(tmp["PC1_core"].var(ddof=1))
        corr = float(tmp[c].corr(tmp["PC1_core"]))
        mets = perf_metrics(tmp[c], rf_daily=_coerce_rf_daily(tmp[c], rf_daily), ann=ann)
        rows.append({
            "asset": c,
            "n_high_val": int(len(tmp)),
            "beta_to_PC1_HIGH": beta,
            "corr_to_PC1_HIGH": corr,
            "ann_vol_HIGH": float(tmp[c].std(ddof=1) * np.sqrt(ann)),
            "CAGR_HIGH": mets.get("CAGR", np.nan),
            "CAGR_excess_HIGH": mets.get("CAGR_excess", np.nan),
            "ShR_HIGH": mets.get("ShR", np.nan),
            "MxDD_HIGH": mets.get("MxDD", np.nan),
            "Martin_HIGH": mets.get("Martin", np.nan),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["beta_to_PC1_HIGH", "corr_to_PC1_HIGH"]).reset_index(drop=True)


def _diag_shrink_cov(R: pd.DataFrame, shrink: float = 0.20) -> np.ndarray:
    cov = pd.DataFrame(R).cov().values.astype(float)
    if cov.size == 0:
        return cov
    diag = np.diag(np.diag(cov))
    shrink = float(np.clip(shrink, 0.0, 1.0))
    cov = (1.0 - shrink) * cov + shrink * diag
    cov = cov + np.eye(cov.shape[0]) * max(1e-10, 1e-8 * np.trace(cov) / max(cov.shape[0], 1))
    return cov


def _cap_and_redistribute(w: pd.Series, max_weight: float, n_iter: int = 100) -> pd.Series:
    w = w.astype(float).copy()
    w = w.clip(lower=0.0)
    if w.sum() <= 0:
        w[:] = 1.0 / len(w)
    else:
        w /= w.sum()
    max_weight = float(max_weight)
    for _ in range(n_iter):
        over = w > max_weight
        if not over.any():
            break
        excess = float((w[over] - max_weight).sum())
        w[over] = max_weight
        under = ~over
        if under.any() and w[under].sum() > 0 and excess > 0:
            w[under] += excess * w[under] / w[under].sum()
        else:
            break
    w = w.clip(lower=0.0)
    return w / w.sum()


def risk_parity_weights(
    cov: np.ndarray,
    labels: list[str],
    max_weight: float = 0.45,
) -> tuple[pd.Series, str]:
    """Long-only risk-parity weights with weight caps; fallback to inverse-vol if optimizer fails."""
    n = len(labels)
    if n == 1:
        return pd.Series([1.0], index=labels, name="weight"), "single_asset"

    diag = np.diag(cov).copy()
    diag[diag <= 0] = np.nan
    iv = 1.0 / np.sqrt(diag)
    iv = np.where(np.isfinite(iv), iv, 0.0)
    if iv.sum() <= 0:
        x0 = np.repeat(1.0 / n, n)
    else:
        x0 = iv / iv.sum()
    if n * max_weight < 1.0 - 1e-12:
        raise ValueError("Infeasible risk-parity max_weight: need n * max_weight >= 1.")

    def rp_obj(w):
        w = np.asarray(w, dtype=float)
        port_var = float(w @ cov @ w)
        if port_var <= 0:
            return 1e9
        mrc = cov @ w
        rc = w * mrc
        target = port_var / len(w)
        return float(np.sum((rc - target) ** 2))

    bounds = [(0.0, float(max_weight)) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)}]

    try:
        from scipy.optimize import minimize
        res = minimize(
            rp_obj,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 2000, "ftol": 1e-14, "disp": False},
        )
        if res.success and np.all(np.isfinite(res.x)):
            w = pd.Series(res.x, index=labels, name="weight").clip(lower=0.0)
            w /= w.sum()
            return w, "risk_parity_slsqp"
    except Exception as exc:
        rp_status = f"risk_parity_exception: {exc}"
    else:
        rp_status = f"risk_parity_fallback: {res.message}"

    vol = np.sqrt(np.maximum(np.diag(cov), 1e-12))
    inv = pd.Series(1.0 / vol, index=labels, name="weight")
    w = inv / inv.sum()
    w = _cap_and_redistribute(w, max_weight=max_weight)
    return w, rp_status + " | inverse_vol_fallback"


def screening_scaled_metrics(
    r: pd.Series,
    rf_daily: pd.Series,
    ann: int = 252,
    target_vol: float | None = None,
) -> dict:
    """
    Screening metrics on a subset of returns:
      - ann_excess_mean (arithmetic)
      - CAGR
      - target-vol Sharpe
    If target_vol is provided, returns are rescaled to the same conditional vol.
    """
    r = pd.Series(r).dropna().astype(float)
    rf = pd.Series(rf_daily).reindex(r.index).ffill().fillna(0.0).astype(float)
    if len(r) < 2:
        return {"n": len(r), "ann_excess_mean": np.nan, "CAGR": np.nan, "ShR_targetvol": np.nan,
                "current_ann_vol": np.nan, "target_vol": target_vol, "scale": np.nan}

    current_vol = float(r.std(ddof=1) * np.sqrt(ann))
    scale = 1.0 if (target_vol is None or not np.isfinite(current_vol) or current_vol <= 0) else float(target_vol) / current_vol

    r_s = r * scale
    rf_s = rf
    excess = r_s - rf_s
    ann_excess_mean = float(excess.mean() * ann)
    cagr = float((1.0 + r_s).prod() ** (ann / len(r_s)) - 1.0)

    denom = float(target_vol) if target_vol is not None else float(r_s.std(ddof=1) * np.sqrt(ann))
    sh_target = ann_excess_mean / denom if np.isfinite(denom) and denom > 0 else np.nan

    return {
        "n": int(len(r_s)),
        "ann_excess_mean": ann_excess_mean,
        "CAGR": cagr,
        "ShR_targetvol": sh_target,
        "current_ann_vol": current_vol,
        "target_vol": float(target_vol) if target_vol is not None else np.nan,
        "scale": scale,
    }


def whole_portfolio_rp_screen(
    core: pd.Series,
    candidates: pd.DataFrame,
    mask: pd.Series,
    rf_daily: pd.Series,
    ann: int = 252,
    target_vol: float | None = None,
    max_weight: float = 0.45,
    cov_shrink: float = 0.20,
    min_obs: int = 30,
) -> tuple[pd.DataFrame, pd.Series, dict]:
    """
    Portfolio-aware screening on validation-HIGH:
      1) build RP weights on [PC1 core + candidate assets];
      2) scale to a common conditional vol;
      3) leave-one-out each candidate asset;
      4) measure contribution to ann excess return / CAGR / target-vol Sharpe.
    """
    core = pd.Series(core).rename("PC1_core").astype(float)
    candidates = pd.DataFrame(candidates).astype(float)
    mask = pd.Series(mask).reindex(core.index).fillna(False).astype(bool)

    df = pd.concat([core, candidates], axis=1).dropna(how="any")
    df_h = df.loc[mask.reindex(df.index).fillna(False)]
    if len(df_h) < max(min_obs, 3):
        raise ValueError(f"Too few validation-HIGH observations for screening: {len(df_h)}")

    all_cols = list(df_h.columns)
    cov_all = _diag_shrink_cov(df_h[all_cols], shrink=cov_shrink)
    w_all, status_all = risk_parity_weights(cov_all, all_cols, max_weight=max_weight)
    port_all = (df_h[all_cols] @ w_all).rename("whole_portfolio_rp")

    target_vol_eff = float(port_all.std(ddof=1) * np.sqrt(ann)) if target_vol is None else float(target_vol)

    full_metrics = screening_scaled_metrics(
        port_all,
        rf_daily=rf_daily.reindex(port_all.index).ffill().fillna(0.0),
        ann=ann,
        target_vol=target_vol_eff,
    )

    rows = []
    candidate_cols = [c for c in all_cols if c != "PC1_core"]
    for asset in candidate_cols:
        cols_loo = [c for c in all_cols if c != asset]
        cov_loo = _diag_shrink_cov(df_h[cols_loo], shrink=cov_shrink)
        w_loo, status_loo = risk_parity_weights(cov_loo, cols_loo, max_weight=max_weight)
        port_loo = (df_h[cols_loo] @ w_loo).rename(f"loo_without_{asset}")

        loo_metrics = screening_scaled_metrics(
            port_loo,
            rf_daily=rf_daily.reindex(port_loo.index).ffill().fillna(0.0),
            ann=ann,
            target_vol=target_vol_eff,
        )

        rows.append({
            "asset": asset,
            "full_weight_in_RP": float(w_all.reindex(all_cols).fillna(0.0).get(asset, 0.0)),
            "full_ann_excess_mean": full_metrics["ann_excess_mean"],
            "full_CAGR": full_metrics["CAGR"],
            "full_ShR_targetvol": full_metrics["ShR_targetvol"],
            "loo_ann_excess_mean": loo_metrics["ann_excess_mean"],
            "loo_CAGR": loo_metrics["CAGR"],
            "loo_ShR_targetvol": loo_metrics["ShR_targetvol"],
            "contrib_excess_ann": full_metrics["ann_excess_mean"] - loo_metrics["ann_excess_mean"],
            "contrib_CAGR": full_metrics["CAGR"] - loo_metrics["CAGR"],
            "contrib_ShR_targetvol": full_metrics["ShR_targetvol"] - loo_metrics["ShR_targetvol"],
            "rp_status_full": status_all,
            "rp_status_loo": status_loo,
        })

    screen = pd.DataFrame(rows).sort_values(
        ["contrib_excess_ann", "contrib_CAGR", "contrib_ShR_targetvol"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    diag = {
        "full_weights": w_all,
        "full_metrics": full_metrics,
        "target_vol": target_vol_eff,
        "n_high_val": int(len(df_h)),
        "screen_returns_high": df_h,
        "port_all_high": port_all,
    }
    return screen, w_all, diag


def select_assets_from_screen(
    screen: pd.DataFrame,
    top_k: int = 5,
    require_positive_excess: bool = True,
    require_nonneg_cagr: bool = False,
) -> list[str]:
    if screen.empty:
        raise ValueError("Empty screening table.")
    tab = screen.copy()
    if require_positive_excess:
        tab = tab.loc[tab["contrib_excess_ann"] > 0]
    if require_nonneg_cagr:
        tab = tab.loc[tab["contrib_CAGR"] >= 0]
    if len(tab) == 0:
        tab = screen.copy()
    return tab.sort_values(
        ["contrib_excess_ann", "contrib_CAGR", "contrib_ShR_targetvol"],
        ascending=[False, False, False]
    )["asset"].head(top_k).tolist()


def max_sharpe_satellite_weights(
    candidate_returns_high: pd.DataFrame,
    rf_daily_high: pd.Series,
    max_weight: float = 0.45,
    cov_shrink: float = 0.20,
) -> tuple[pd.Series, dict]:
    """Long-only capped max-Sharpe portfolio on selected satellite assets in validation-HIGH."""
    R = pd.DataFrame(candidate_returns_high).dropna(how="any").astype(float)
    if R.empty:
        raise ValueError("Empty candidate_returns_high in max_sharpe_satellite_weights.")
    labels = list(R.columns)
    n = len(labels)
    if n * max_weight < 1.0 - 1e-12:
        raise ValueError("Infeasible max_weight for max-Sharpe satellite: need n * max_weight >= 1.")

    rf = pd.Series(rf_daily_high).reindex(R.index).ffill().fillna(0.0).astype(float)
    mu = R.sub(rf, axis=0).mean().reindex(labels).fillna(0.0).values * 252
    Sigma = _diag_shrink_cov(R[labels], shrink=cov_shrink)

    if n == 1:
        w = pd.Series([1.0], index=labels, name="satellite_weight")
        vol = float(np.sqrt(Sigma[0, 0]) * np.sqrt(252))
        diag = {
            "opt_status": "single_asset",
            "ann_excess_return": float(mu[0]),
            "ann_vol": vol,
            "ann_sharpe": float(mu[0] / vol) if vol > 0 else np.nan,
        }
        return w, diag

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-12))
    inv = pd.Series(1.0 / vol, index=labels)
    x0 = (inv / inv.sum()).values

    def neg_sharpe(w):
        w = np.asarray(w, dtype=float)
        ex = float(w @ mu)
        var = float(w @ Sigma @ w)
        if var <= 0:
            return 1e9
        return -ex / np.sqrt(var)

    bounds = [(0.0, float(max_weight)) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)}]

    try:
        from scipy.optimize import minimize
        res = minimize(
            neg_sharpe,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 2000, "ftol": 1e-14, "disp": False},
        )
        if res.success and np.all(np.isfinite(res.x)):
            w = pd.Series(res.x, index=labels, name="satellite_weight").clip(lower=0.0)
            w /= w.sum()
            status = "max_sharpe_slsqp"
        else:
            w = (inv / inv.sum()).rename("satellite_weight")
            w = _cap_and_redistribute(w, max_weight=max_weight)
            status = f"max_sharpe_fallback_inverse_vol: {res.message}"
    except Exception as exc:
        w = (inv / inv.sum()).rename("satellite_weight")
        w = _cap_and_redistribute(w, max_weight=max_weight)
        status = f"max_sharpe_exception_inverse_vol: {exc}"

    ann_ex = float(w.values @ mu)
    ann_vol = float(np.sqrt(w.values @ Sigma @ w.values))
    diag = {
        "opt_status": status,
        "ann_excess_return": ann_ex,
        "ann_vol": ann_vol,
        "ann_sharpe": float(ann_ex / ann_vol) if ann_vol > 0 else np.nan,
        "mu_excess": pd.Series(mu, index=labels),
    }
    return w, diag




def min_variance_satellite_weights(
    candidate_returns_high: pd.DataFrame,
    rf_daily_high: pd.Series,
    max_weight: float = 0.45,
    cov_shrink: float = 0.20,
) -> tuple[pd.Series, dict]:
    """Long-only capped MVP on selected satellite assets in validation-HIGH."""
    R = pd.DataFrame(candidate_returns_high).dropna(how="any").astype(float)
    if R.empty:
        raise ValueError("Empty candidate_returns_high in min_variance_satellite_weights.")
    labels = list(R.columns)
    n = len(labels)
    if n * max_weight < 1.0 - 1e-12:
        raise ValueError("Infeasible max_weight for MVP satellite: need n * max_weight >= 1.")

    rf = pd.Series(rf_daily_high).reindex(R.index).ffill().fillna(0.0).astype(float)
    mu = R.sub(rf, axis=0).mean().reindex(labels).fillna(0.0).values * 252
    Sigma = _diag_shrink_cov(R[labels], shrink=cov_shrink)

    if n == 1:
        w = pd.Series([1.0], index=labels, name="satellite_weight")
        vol = float(np.sqrt(Sigma[0, 0]) * np.sqrt(252))
        diag = {
            "opt_status": "single_asset",
            "ann_excess_return": float(mu[0]),
            "ann_vol": vol,
            "ann_sharpe": float(mu[0] / vol) if vol > 0 else np.nan,
            "mu_excess": pd.Series(mu, index=labels),
        }
        return w, diag

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-12))
    inv = pd.Series(1.0 / vol, index=labels)
    x0 = (inv / inv.sum()).values

    def var_obj(w):
        w = np.asarray(w, dtype=float)
        return float(w @ Sigma @ w)

    bounds = [(0.0, float(max_weight)) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)}]

    try:
        from scipy.optimize import minimize
        res = minimize(
            var_obj,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 2000, "ftol": 1e-14, "disp": False},
        )
        if res.success and np.all(np.isfinite(res.x)):
            w = pd.Series(res.x, index=labels, name="satellite_weight").clip(lower=0.0)
            w /= w.sum()
            status = "mvp_slsqp"
        else:
            w = (inv / inv.sum()).rename("satellite_weight")
            w = _cap_and_redistribute(w, max_weight=max_weight)
            status = f"mvp_fallback_inverse_vol: {res.message}"
    except Exception as exc:
        w = (inv / inv.sum()).rename("satellite_weight")
        w = _cap_and_redistribute(w, max_weight=max_weight)
        status = f"mvp_exception_inverse_vol: {exc}"

    ann_ex = float(w.values @ mu)
    ann_vol = float(np.sqrt(max(w.values @ Sigma @ w.values, 0.0)))
    diag = {
        "opt_status": status,
        "ann_excess_return": ann_ex,
        "ann_vol": ann_vol,
        "ann_sharpe": float(ann_ex / ann_vol) if ann_vol > 0 else np.nan,
        "mu_excess": pd.Series(mu, index=labels),
    }
    return w, diag


def frontier_blended_satellite_return(
    candidate_returns: pd.DataFrame,
    w_tan: pd.Series,
    w_mvp: pd.Series,
    si: pd.Series,
) -> tuple[pd.Series, pd.DataFrame]:
    """Dynamic satellite return using THE SAME stress indicator SI for the inner frontier motion.

    Satellite sleeve:
        w_sat,t = (1 - SI_{t-1}) * w_tan + SI_{t-1} * w_mvp
    """
    R = pd.DataFrame(candidate_returns).copy()
    cols = list(pd.Index(w_tan.index).intersection(w_mvp.index).intersection(R.columns))
    if len(cols) == 0:
        raise ValueError("No common asset columns for frontier_blended_satellite_return.")
    R = R[cols].astype(float).fillna(0.0)

    w_tan = w_tan.reindex(cols).fillna(0.0).astype(float)
    w_mvp = w_mvp.reindex(cols).fillna(0.0).astype(float)

    if float(w_tan.sum()) <= 0:
        raise ValueError("Tangency satellite weights sum to zero.")
    if float(w_mvp.sum()) <= 0:
        raise ValueError("MVP satellite weights sum to zero.")

    w_tan = w_tan / w_tan.sum()
    w_mvp = w_mvp / w_mvp.sum()

    si_ = pd.Series(si).reindex(R.index).fillna(0.0).clip(0.0, 1.0)
    si_lag = si_.shift(1).fillna(0.0).clip(0.0, 1.0)

    w_df = pd.DataFrame(index=R.index, columns=cols, dtype=float)
    for c in cols:
        w_df[c] = (1.0 - si_lag) * float(w_tan[c]) + si_lag * float(w_mvp[c])

    sat_ret = (R * w_df).sum(axis=1).rename("Satellite_frontier_dynamic")
    diag = w_df.copy()
    diag["SI"] = si_
    diag["SI_lag"] = si_lag
    diag["satellite_return"] = sat_ret
    diag["satellite_internal_turnover"] = 0.5 * w_df.diff().abs().sum(axis=1).fillna(0.0)
    return sat_ret, diag


def core_satellite_frontier_strategy(
    core_net: pd.Series,
    candidate_returns: pd.DataFrame,
    w_tan: pd.Series,
    w_mvp: pd.Series,
    si: pd.Series,
    tc_bps: float = 10.0,
) -> tuple[pd.Series, pd.DataFrame]:
    """Strategy with dynamic satellite frontier driven by the SAME stress indicator SI.

    Outer allocation:
        (1 - SI_{t-1}) * core + SI_{t-1} * satellite_t

    Inner satellite allocation:
        w_sat,t = (1 - SI_{t-1}) * w_tan + SI_{t-1} * w_mvp
    """
    sat_ret, sat_diag = frontier_blended_satellite_return(
        candidate_returns=candidate_returns,
        w_tan=w_tan,
        w_mvp=w_mvp,
        si=si,
    )

    df = pd.concat(
        [core_net.rename("core"), sat_ret.rename("satellite"), si.rename("SI")],
        axis=1,
    ).dropna(subset=["core", "satellite"])

    df["SI_lag"] = df["SI"].shift(1).fillna(0.0).clip(0.0, 1.0)
    df["w_core"] = 1.0 - df["SI_lag"]
    df["w_satellite"] = df["SI_lag"]

    gross = df["w_core"] * df["core"] + df["w_satellite"] * df["satellite"]

    overlay_turn = df["SI_lag"].diff().abs().fillna(0.0)
    sat_internal_turn = sat_diag["satellite_internal_turnover"].reindex(df.index).fillna(0.0)

    # Approximate total daily turnover:
    #   - changing outer sleeve weight between core and satellite
    #   - changing internal satellite composition, scaled by current satellite sleeve size
    total_turn = overlay_turn + df["SI_lag"] * sat_internal_turn

    cost = (float(tc_bps) / 1e4) * total_turn
    net = (gross - cost).rename("Core_SI_FrontierSatellite_net")

    diag = df.copy()
    diag["gross_return"] = gross
    diag["overlay_turnover"] = overlay_turn
    diag["satellite_internal_turnover"] = sat_internal_turn
    diag["total_turnover"] = total_turn
    diag["overlay_cost"] = cost
    diag["net_return"] = net
    diag["inner_SI"] = sat_diag["SI"].reindex(df.index).fillna(0.0)
    diag["inner_SI_lag"] = sat_diag["SI_lag"].reindex(df.index).fillna(0.0)
    return net, diag




def core_satellite_tan_rf_strategy(
    core_net: pd.Series,
    candidate_returns: pd.DataFrame,
    w_tan: pd.Series,
    rf_ret: pd.Series,
    si: pd.Series,
    rf_gamma: float = 1.0,
    tc_bps: float = 10.0,
) -> tuple[pd.Series, pd.DataFrame]:
    """Dynamic core-satellite strategy with nonlinear Tangency -> RF motion.

    Outer allocation:
        Core weight            = 1 - SI_{t-1}
        Satellite total weight = SI_{t-1}

    Inner satellite:
        h_gamma(SI) = SI ** rf_gamma
        Satellite_t = (1 - h_gamma(SI_{t-1})) * Tangency + h_gamma(SI_{t-1}) * RF

    Therefore total-portfolio sleeve weights are:
        Core               = 1 - SI
        Tangency sleeve    = SI * (1 - SI ** rf_gamma)
        RF endpoint        = SI * (SI ** rf_gamma)
    """
    R = pd.DataFrame(candidate_returns).copy()
    cols = list(pd.Index(w_tan.index).intersection(R.columns))
    if len(cols) == 0:
        raise ValueError("No common asset columns for tangency satellite.")
    R = R[cols].astype(float).fillna(0.0)

    w_tan = w_tan.reindex(cols).fillna(0.0).astype(float)
    if float(w_tan.sum()) <= 0:
        raise ValueError("Tangency satellite weights sum to zero.")
    w_tan = w_tan / w_tan.sum()

    df = pd.concat(
        [
            pd.Series(core_net).rename("core"),
            R,
            pd.Series(rf_ret).rename("rf"),
            pd.Series(si).rename("SI"),
        ],
        axis=1,
    ).dropna(subset=["core", "rf"])

    df["SI_lag"] = df["SI"].shift(1).fillna(0.0).clip(0.0, 1.0)
    df["h_rf"] = df["SI_lag"].pow(float(rf_gamma)).clip(0.0, 1.0)

    # Sleeve weights in the TOTAL portfolio.
    eff_w_core = 1.0 - df["SI_lag"]
    eff_w_tan_total = df["SI_lag"] * (1.0 - df["h_rf"])
    eff_w_rf = df["SI_lag"] * df["h_rf"]

    eff_w_assets = pd.DataFrame(index=df.index, columns=cols, dtype=float)
    for c in cols:
        eff_w_assets[c] = eff_w_tan_total * float(w_tan[c])

    risky_sat_ret = (df[cols] * pd.DataFrame(np.tile(w_tan.values, (len(df), 1)), index=df.index, columns=cols)).sum(axis=1)
    sat_ret = ((1.0 - df["h_rf"]) * risky_sat_ret + df["h_rf"] * df["rf"]).rename("Satellite_tan_rf_dynamic")

    gross = eff_w_core * df["core"] + (eff_w_assets * df[cols]).sum(axis=1) + eff_w_rf * df["rf"]

    # Exact sleeve-space turnover across core + risky satellite assets + RF.
    weight_panel = eff_w_assets.copy()
    weight_panel["Core"] = eff_w_core
    weight_panel["RF_endpoint"] = eff_w_rf
    weight_panel = weight_panel[["Core"] + cols + ["RF_endpoint"]]
    total_turn = 0.5 * weight_panel.diff().abs().sum(axis=1).fillna(0.0)

    cost = (float(tc_bps) / 1e4) * total_turn
    net = (gross - cost).rename("Core_SI_TanRF_Satellite_net")

    diag = pd.DataFrame(index=df.index)
    diag["core"] = df["core"]
    diag["risky_sat_ret"] = risky_sat_ret
    diag["rf"] = df["rf"]
    diag["SI_lag"] = df["SI_lag"]
    diag["h_rf"] = df["h_rf"]
    diag["sat_ret"] = sat_ret
    diag["w_core"] = eff_w_core
    diag["w_satellite_total"] = df["SI_lag"]
    diag["w_tangency_sleeve"] = eff_w_tan_total
    diag["w_rf_endpoint"] = eff_w_rf
    diag["overlay_turnover"] = total_turn
    diag["overlay_cost"] = cost
    diag["gross_ret"] = gross
    diag["net_ret"] = net
    diag["rf_gamma"] = float(rf_gamma)
    for c in cols:
        diag[f"w_{c}"] = eff_w_assets[c]
    return net, diag





def core_satellite_nonlinear_both_strategy(
    core_net: pd.Series,
    candidate_returns: pd.DataFrame,
    w_tan: pd.Series,
    rf_ret: pd.Series,
    si: pd.Series,
    outer_gamma: float = 1.0,
    rf_gamma: float = 1.0,
    tc_bps: float = 10.0,
) -> tuple[pd.Series, pd.DataFrame]:
    """Dynamic core-satellite strategy with NONLINEAR outer and NONLINEAR inner motion.

    Outer allocation:
        g_core(SI) = SI ** outer_gamma
        Core weight            = 1 - g_core(SI_{t-1})
        Satellite total weight = g_core(SI_{t-1})

    Inner satellite:
        h_rf(SI) = SI ** rf_gamma
        Satellite_t = (1 - h_rf(SI_{t-1})) * Tangency + h_rf(SI_{t-1}) * RF

    Therefore total-portfolio sleeve weights are:
        Core               = 1 - SI ** outer_gamma
        Tangency sleeve    = (SI ** outer_gamma) * (1 - SI ** rf_gamma)
        RF endpoint        = (SI ** outer_gamma) * (SI ** rf_gamma)
    """
    R = pd.DataFrame(candidate_returns).copy()
    cols = list(pd.Index(w_tan.index).intersection(R.columns))
    if len(cols) == 0:
        raise ValueError("No common asset columns for tangency satellite.")
    R = R[cols].astype(float).fillna(0.0)

    w_tan = w_tan.reindex(cols).fillna(0.0).astype(float)
    if float(w_tan.sum()) <= 0:
        raise ValueError("Tangency satellite weights sum to zero.")
    w_tan = w_tan / w_tan.sum()

    df = pd.concat(
        [
            pd.Series(core_net).rename("core"),
            R,
            pd.Series(rf_ret).rename("rf"),
            pd.Series(si).rename("SI"),
        ],
        axis=1,
    ).dropna(subset=["core", "rf"])

    df["SI_lag"] = df["SI"].shift(1).fillna(0.0).clip(0.0, 1.0)
    df["g_core"] = df["SI_lag"].pow(float(outer_gamma)).clip(0.0, 1.0)
    df["h_rf"] = df["SI_lag"].pow(float(rf_gamma)).clip(0.0, 1.0)

    eff_w_core = 1.0 - df["g_core"]
    eff_w_tan_total = df["g_core"] * (1.0 - df["h_rf"])
    eff_w_rf = df["g_core"] * df["h_rf"]

    eff_w_assets = pd.DataFrame(index=df.index, columns=cols, dtype=float)
    for c in cols:
        eff_w_assets[c] = eff_w_tan_total * float(w_tan[c])

    risky_sat_ret = (
        df[cols] * pd.DataFrame(np.tile(w_tan.values, (len(df), 1)), index=df.index, columns=cols)
    ).sum(axis=1)
    sat_ret = ((1.0 - df["h_rf"]) * risky_sat_ret + df["h_rf"] * df["rf"]).rename("Satellite_tan_rf_dynamic_nonlin")

    gross = eff_w_core * df["core"] + (eff_w_assets * df[cols]).sum(axis=1) + eff_w_rf * df["rf"]

    weight_panel = eff_w_assets.copy()
    weight_panel["Core"] = eff_w_core
    weight_panel["RF_endpoint"] = eff_w_rf
    weight_panel = weight_panel[["Core"] + cols + ["RF_endpoint"]]

    turn = weight_panel.diff().abs().sum(axis=1).fillna(0.0)
    cost = (float(tc_bps) / 1e4) * turn
    net = (gross - cost).rename("Core_SIgamma_SatelliteTanRF_RFgamma_net")

    diag = df.copy()
    diag["satellite_ret"] = sat_ret
    diag["w_core"] = eff_w_core
    diag["w_tan_total"] = eff_w_tan_total
    diag["w_rf_endpoint"] = eff_w_rf
    diag["gross_return"] = gross
    diag["overlay_turnover"] = turn
    diag["overlay_cost"] = cost
    diag["net_return"] = net
    diag["outer_gamma"] = float(outer_gamma)
    diag["rf_gamma"] = float(rf_gamma)
    diag["risky_sat_ret"] = risky_sat_ret

    for c in cols:
        diag[f"w_{c}"] = eff_w_assets[c]

    return net, diag


def core_satellite_nonlinear_outer_strategy(
    core_net: pd.Series,
    satellite_ret: pd.Series,
    si: pd.Series,
    outer_gamma: float = 1.0,
    tc_bps: float = 10.0,
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Nonlinear OUTER allocation with lagged stress intensity:
        g_gamma(SI) = SI ** outer_gamma

        total return =
            (1 - g_gamma(SI_{t-1})) * core_net
            + g_gamma(SI_{t-1}) * satellite_ret
            - allocation_turnover_cost

    This version keeps the tangency satellite static and only changes the
    speed of rotation between Core and Satellite.
    """
    df = pd.concat(
        [core_net.rename("core"), satellite_ret.rename("satellite"), si.rename("SI_raw")],
        axis=1,
    ).dropna(subset=["core", "satellite"])

    df["SI_raw_lag"] = df["SI_raw"].shift(1).fillna(0.0).clip(0.0, 1.0)
    df["g_SI_lag"] = df["SI_raw_lag"].pow(float(outer_gamma)).clip(0.0, 1.0)

    df["w_core"] = 1.0 - df["g_SI_lag"]
    df["w_satellite"] = df["g_SI_lag"]

    gross = df["w_core"] * df["core"] + df["w_satellite"] * df["satellite"]

    turn = df["g_SI_lag"].diff().abs().fillna(0.0)
    cost = (float(tc_bps) / 1e4) * turn
    net = (gross - cost).rename("Core_SIgamma_SatelliteTangency_net")

    diag = df.copy()
    diag["gross_return"] = gross
    diag["overlay_turnover"] = turn
    diag["overlay_cost"] = cost
    diag["net_return"] = net
    diag["outer_gamma"] = float(outer_gamma)
    return net, diag


def core_satellite_strategy(

    core_net: pd.Series,
    satellite_ret: pd.Series,
    si: pd.Series,
    tc_bps: float = 10.0,
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Strategy return with lagged SI:
        (1-SI_{t-1}) * PC1_core_net + SI_{t-1} * satellite_return - allocation_turnover_cost.
    """
    df = pd.concat(
        [core_net.rename("core"), satellite_ret.rename("satellite"), si.rename("SI")],
        axis=1,
    ).dropna(subset=["core", "satellite"])
    df["SI_lag"] = df["SI"].shift(1).fillna(0.0).clip(0.0, 1.0)
    df["w_core"] = 1.0 - df["SI_lag"]
    df["w_satellite"] = df["SI_lag"]
    gross = df["w_core"] * df["core"] + df["w_satellite"] * df["satellite"]

    turn = df["SI_lag"].diff().abs().fillna(0.0)
    cost = (float(tc_bps) / 1e4) * turn
    net = (gross - cost).rename("Core_SI_Satellite_net")

    diag = df.copy()
    diag["gross_return"] = gross
    diag["overlay_turnover"] = turn
    diag["overlay_cost"] = cost
    diag["net_return"] = net
    return net, diag


def metrics_table_for_series(series_dict: dict[str, pd.Series], rf_daily: pd.Series | float | None = 0.0) -> pd.DataFrame:
    rows = []
    for name, r in series_dict.items():
        r = pd.Series(r).dropna()
        rf_sub = rf_daily.reindex(r.index).ffill().fillna(0.0) if hasattr(rf_daily, "reindex") else rf_daily
        row = perf_metrics(r, rf_daily=rf_sub)
        row["strategy"] = name
        rows.append(row)
    if len(rows) == 0:
        return pd.DataFrame()
    out = pd.DataFrame(rows).set_index("strategy")
    cols = ["n", "CAGR", "CAGR_excess", "RF_CAGR", "MxDD", "ShR", "SoR", "Calmar", "Martin", "Ulcer"]
    return out[[c for c in cols if c in out.columns]]


def paired_bootstrap_metric_diffs(
    df_returns: pd.DataFrame,
    target: str,
    benchmarks: list[str],
    rf_daily: pd.Series | float | None,
    metrics: tuple[str, ...] = ("CAGR", "CAGR_excess", "ShR", "SoR", "Martin", "MxDD"),
    n_boot: int = 1000,
    avg_block_len: float = 43.0,
    seed: int = 7,
) -> pd.DataFrame:
    """Stationary-bootstrap differences target minus benchmark using paired rows."""
    df = df_returns.dropna(how="any").copy()
    if len(df) < 50:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    rows = []
    for b in benchmarks:
        diffs = {m: [] for m in metrics}
        for _ in range(n_boot):
            ii = stationary_bootstrap_indices(len(df), avg_block_len=avg_block_len, rng=rng)
            sim = df.iloc[ii].reset_index(drop=True)
            if hasattr(rf_daily, "reindex"):
                rf_sim = rf_daily.reindex(df.index).ffill().fillna(0.0).iloc[ii].reset_index(drop=True)
            else:
                rf_sim = rf_daily
            mt = perf_metrics(sim[target], rf_daily=rf_sim)
            mb = perf_metrics(sim[b], rf_daily=rf_sim)
            for m in metrics:
                diffs[m].append(mt.get(m, np.nan) - mb.get(m, np.nan))
        for m, vals in diffs.items():
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            rows.append({
                "target": target,
                "benchmark": b,
                "metric": m,
                "mean_diff": float(np.mean(vals)) if len(vals) else np.nan,
                "p05_diff": float(np.quantile(vals, 0.05)) if len(vals) else np.nan,
                "p50_diff": float(np.quantile(vals, 0.50)) if len(vals) else np.nan,
                "p95_diff": float(np.quantile(vals, 0.95)) if len(vals) else np.nan,
                "P_target_gt_benchmark": float(np.mean(vals > 0)) if len(vals) else np.nan,
                "n_boot": int(len(vals)),
            })
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# Download external asset-class proxies, run portfolio-aware screening, and build satellite
# ============================================================

if F_wf_net.empty:
    raise RuntimeError("Walk-forward PC1 returns are empty. Run the walk-forward section before this cell.")

HIGH_LABEL = "HIGH" if "HIGH" in REGIME_ORDER else REGIME_ORDER[-1]
print("Satellite allocation high/stress label:", HIGH_LABEL)

# Core: walk-forward PC1 already net of stock-level rebalancing costs.
core_pc1 = F_wf_net["PC1"].dropna().rename("PC1_core_net")
core_idx = pd.DatetimeIndex(core_pc1.index)

reg_core = reg.reindex(core_idx).dropna()
core_pc1 = core_pc1.reindex(reg_core.index).dropna()
reg_core = reg_core.reindex(core_pc1.index)
rf_core = rf_daily_hist.reindex(core_pc1.index).ffill().fillna(0.0)

# Resolve validation and frozen final OOS split from available tradable index.
sat_val_start, sat_val_end, sat_final_start, split_mode = resolve_core_satellite_splits(
    core_pc1.index,
    validation_start=SAT_VALIDATION_START,
    validation_end=SAT_VALIDATION_END,
    final_oos_start=SAT_FINAL_OOS_START,
    validation_frac=SAT_VALIDATION_FRAC,
    min_validation_days=SAT_MIN_VALIDATION_DAYS,
    min_final_days=SAT_MIN_FINAL_DAYS,
)

print(f"Satellite split mode: {split_mode}")
print(f"Validation range: {sat_val_start.date()} -> {sat_val_end.date()}")
print(f"Frozen final OOS start: {sat_final_start.date()}")

# Stress intensity: empirical percentile of IV score within historical HIGH state.
SI_BY_GAMMA = {}
for gamma in SI_GAMMA_GRID:
    si = build_high_regime_si(
        iv_score=z_daily,
        regime=reg,
        high_label=HIGH_LABEL,
        gamma=gamma,
        min_obs=SI_CONDITIONAL_MIN_OBS,
    ).reindex(core_pc1.index).fillna(0.0)
    SI_BY_GAMMA[gamma] = si

si_diag = pd.DataFrame(SI_BY_GAMMA)
si_diag.columns = [f"gamma_{g:g}" for g in SI_BY_GAMMA.keys()]
print("Stress-intensity diagnostics on tradable index:")
display(si_diag.describe().T[["mean", "50%", "75%", "max"]])
print("Average SI by regime:")
display(pd.concat([reg_core.rename("regime"), si_diag], axis=1).groupby("regime").mean())

# Download external asset-class returns.
sat_ret_raw = download_asset_class_returns(SATELLITE_ASSET_CLASS_TICKERS, start=START, end=END)
if sat_ret_raw.empty:
    raise RuntimeError("No satellite asset-class returns downloaded. Check tickers / internet connection.")

sat_ret = sat_ret_raw.reindex(core_pc1.index)
coverage = sat_ret.notna().mean()
keep_cols = coverage[coverage >= SAT_MIN_COVERAGE].index.tolist()
if len(keep_cols) == 0:
    raise RuntimeError("No satellite asset-class series passed the coverage filter.")

sat_ret = sat_ret[keep_cols].fillna(0.0)
print("Downloaded/usable satellite asset-class return columns:")
print(list(sat_ret.columns))
print("Satellite returns shape after alignment/filtering:", sat_ret.shape)
print("Satellite data coverage before fill:")
display(coverage.sort_values(ascending=False).to_frame("coverage"))

# Validation / final masks.
val_mask = pd.Series((core_pc1.index >= sat_val_start) & (core_pc1.index <= sat_val_end), index=core_pc1.index)
final_mask = pd.Series(core_pc1.index >= sat_final_start, index=core_pc1.index)
high_mask_val = val_mask & reg_core.eq(HIGH_LABEL)

print("Validation observations:", int(val_mask.sum()), "| HIGH validation observations:", int(high_mask_val.sum()))
print("Final OOS observations:", int(final_mask.sum()))

if int(high_mask_val.sum()) < SAT_MIN_HIGH_OBS:
    raise RuntimeError(
        f"Too few HIGH observations in validation ({int(high_mask_val.sum())}). "
        "Use a longer validation window or lower SAT_MIN_HIGH_OBS."
    )

beta_table = conditional_beta_corr_table(
    core=core_pc1,
    candidates=sat_ret.reindex(core_pc1.index),
    mask=high_mask_val,
    rf_daily=rf_core,
    min_obs=SAT_MIN_HIGH_OBS,
)
print("Candidate diagnostics in validation HIGH state:")
display(beta_table)

# Candidate universe for screening: exclude cash-like sleeves from the risky satellite by default.
sat_candidates = sat_ret.copy()
if not SAT_SCREEN_USE_CASHLIKE:
    sat_candidates = sat_candidates[[c for c in sat_candidates.columns if c not in SAT_CASHLIKE_COLS]]

# Portfolio-aware screening: whole portfolio [PC1 + candidates], RP, same conditional vol plane.
screen_table, rp_full_weights, screen_diag = whole_portfolio_rp_screen(
    core=core_pc1,
    candidates=sat_candidates.reindex(core_pc1.index),
    mask=high_mask_val,
    rf_daily=rf_core,
    target_vol=SAT_SCREEN_TARGET_VOL,
    max_weight=SAT_SCREEN_MAX_WEIGHT,
    cov_shrink=SAT_SCREEN_COV_SHRINK,
    min_obs=SAT_MIN_HIGH_OBS,
)

print("Whole-portfolio RP full weights on validation HIGH (PC1 + candidate asset classes):")
display(rp_full_weights.to_frame("weight"))
print("Screening reference metrics at common conditional risk:")
display(pd.Series(screen_diag["full_metrics"]).to_frame("value"))
print("Portfolio-aware leave-one-out screening table:")
display(screen_table)

selected_assets = select_assets_from_screen(
    screen=screen_table,
    top_k=SAT_SCREEN_TOP_K,
    require_positive_excess=SAT_SCREEN_REQUIRE_POSITIVE_EXCESS,
    require_nonneg_cagr=SAT_SCREEN_REQUIRE_NONNEG_CAGR,
)

print("Selected risky asset classes from screening:")
print(selected_assets)

satellite_universe = sat_ret.reindex(core_pc1.index)[selected_assets].copy()
if not SAT_MAXSHARPE_ALLOW_CASHLIKE:
    satellite_universe = satellite_universe[[c for c in satellite_universe.columns if c not in SAT_CASHLIKE_COLS]]

if satellite_universe.shape[1] == 0:
    raise RuntimeError("Satellite universe empty after screening / cash-like exclusion settings.")

# Build the risky-satellite tangency endpoint on the selected assets only, using validation-HIGH.
sat_high = satellite_universe.loc[high_mask_val.reindex(satellite_universe.index).fillna(False)].dropna(how="any")
rf_high = rf_core.reindex(sat_high.index).ffill().fillna(0.0)

sat_w_tan, sat_diag_tan = max_sharpe_satellite_weights(
    candidate_returns_high=sat_high,
    rf_daily_high=rf_high,
    max_weight=SAT_MAXSHARPE_MAX_WEIGHT,
    cov_shrink=SAT_MAXSHARPE_COV_SHRINK,
)

print("\nSelected validation-frozen risky-satellite weights (tangency / max-Sharpe):")
display(sat_w_tan.to_frame("w_tan"))

print("Tangency satellite optimizer status:", sat_diag_tan.get("opt_status"))
display(pd.Series({
    "tan_ann_excess_ret_HIGH": sat_diag_tan.get("ann_excess_return"),
    "tan_ann_vol_HIGH": sat_diag_tan.get("ann_vol"),
    "tan_ann_sharpe_HIGH": sat_diag_tan.get("ann_sharpe"),
}).to_frame("value"))

satellite_basket_ret_tan = (satellite_universe[sat_w_tan.index].fillna(0.0) @ sat_w_tan).rename("Satellite_tangency")

basket_diag_tan = conditional_beta_corr_table(
    core=core_pc1,
    candidates=satellite_basket_ret_tan.to_frame(),
    mask=high_mask_val,
    rf_daily=rf_core,
    min_obs=SAT_MIN_HIGH_OBS,
)
print("Tangency risky-satellite diagnostics in validation HIGH:")
display(basket_diag_tan)

# Keep explicit cash-like / RF endpoint separate from the risky satellite.
cashlike_cols_present = [c for c in SAT_CASHLIKE_COLS if c in sat_ret.columns]
cashlike_diag = conditional_beta_corr_table(
    core=core_pc1,
    candidates=pd.concat(
        [
            sat_ret[cashlike_cols_present].reindex(core_pc1.index).fillna(0.0),
            rf_core.rename("RF_proxy").to_frame()
        ],
        axis=1,
    ),
    mask=high_mask_val,
    rf_daily=rf_core,
    min_obs=SAT_MIN_HIGH_OBS,
)
print("Cash-like / RF endpoint diagnostics in validation HIGH:")
display(cashlike_diag)

SATELLITE_BUILD_OBJECTS = {
    "core_pc1": core_pc1,
    "reg_core": reg_core,
    "rf_core": rf_core,
    "sat_ret": sat_ret,
    "beta_table": beta_table,
    "screen_table": screen_table,
    "screen_full_weights": rp_full_weights,
    "screen_diag": screen_diag,
    "selected_assets": selected_assets,
    "satellite_weights_tan": sat_w_tan,
    "satellite_basket_ret_tan": satellite_basket_ret_tan,
    "satellite_diag_tan": sat_diag_tan,
    "cashlike_diag": cashlike_diag,
    "si_by_gamma": SI_BY_GAMMA,
    "validation_mask": val_mask,
    "final_mask": final_mask,
    "high_mask_validation": high_mask_val,
    "split": {
        "mode": split_mode,
        "validation_start": sat_val_start,
        "validation_end": sat_val_end,
        "final_oos_start": sat_final_start,
    },
}


In [ ]:
# ============================================================
# Core-satellite strategy: LOW = 100% PC1; HIGH = SI-scaled satellite allocation.
#
# Families kept together on purpose:
#   1) Static tangency satellite
#   2) Nonlinear OUTER motion between PC1 and the static tangency satellite
#   3) Linear inner Tangency -> RF family
#   4) Nonlinear BOTH: OUTER Core -> Satellite and INNER Tangency -> RF
#   5) Benchmark PC1 -> RF family
# ============================================================

candidate_strategy_returns = {}
candidate_strategy_diags = {}

# 1) Static tangency-satellite family kept as the baseline.
for gamma, si in SI_BY_GAMMA.items():
    strat_tan, diag_tan = core_satellite_strategy(
        core_net=core_pc1,
        satellite_ret=satellite_basket_ret_tan,
        si=si,
        tc_bps=SAT_OVERLAY_TC_BPS,
    )
    name = f"Core_PC1_to_SatelliteTan_SI_gamma_{gamma:g}_net"
    candidate_strategy_returns[name] = strat_tan.rename(name)
    candidate_strategy_diags[name] = diag_tan

# 2) Nonlinear OUTER motion between PC1 and the static tangency satellite.
for gamma_si, si in SI_BY_GAMMA.items():
    for gamma_core in CORE_SAT_GAMMA_GRID:
        strat_nl, diag_nl = core_satellite_nonlinear_outer_strategy(
            core_net=core_pc1,
            satellite_ret=satellite_basket_ret_tan,
            si=si,
            outer_gamma=gamma_core,
            tc_bps=SAT_OVERLAY_TC_BPS,
        )
        name = f"Core_PC1_to_SatelliteTan_SIgamma_{gamma_si:g}_CoreGamma_{gamma_core:g}_net"
        candidate_strategy_returns[name] = strat_nl.rename(name)
        candidate_strategy_diags[name] = diag_nl

# 3) Linear inner Tangency -> RF family.
#    SAME SI drives both:
#      - outer Core vs Satellite split (linear in SI)
#      - inner Tangency -> RF motion (linear in SI, i.e. rf_gamma = 1)
for gamma_si, si in SI_BY_GAMMA.items():
    strat_dyn, diag_dyn = core_satellite_tan_rf_strategy(
        core_net=core_pc1,
        candidate_returns=satellite_universe,
        w_tan=sat_w_tan,
        rf_ret=rf_core,
        si=si,
        rf_gamma=1.0,
        tc_bps=SAT_OVERLAY_TC_BPS,
    )
    name = f"Core_PC1_to_SatelliteTanRF_SI_gamma_{gamma_si:g}_net"
    candidate_strategy_returns[name] = strat_dyn.rename(name)
    candidate_strategy_diags[name] = diag_dyn

# 4) Nonlinear BOTH: nonlinear outer Core -> Satellite and nonlinear inner Tangency -> RF.
for gamma_si, si in SI_BY_GAMMA.items():
    for gamma_core in CORE_SAT_GAMMA_GRID:
        for gamma_rf in RF_INNER_GAMMA_GRID:
            strat_both, diag_both = core_satellite_nonlinear_both_strategy(
                core_net=core_pc1,
                candidate_returns=satellite_universe,
                w_tan=sat_w_tan,
                rf_ret=rf_core,
                si=si,
                outer_gamma=gamma_core,
                rf_gamma=gamma_rf,
                tc_bps=SAT_OVERLAY_TC_BPS,
            )
            name = (
                f"Core_PC1_to_SatelliteTanRF_"
                f"SIgamma_{gamma_si:g}_CoreGamma_{gamma_core:g}_RFgamma_{gamma_rf:g}_net"
            )
            candidate_strategy_returns[name] = strat_both.rename(name)
            candidate_strategy_diags[name] = diag_both

# 5) Benchmark family: PC1 -> RF proxy
for gamma, si in SI_BY_GAMMA.items():
    strat_rf, diag_rf = core_satellite_strategy(
        core_net=core_pc1,
        satellite_ret=rf_core.rename("RF_proxy"),
        si=si,
        tc_bps=SAT_OVERLAY_TC_BPS,
    )
    name = f"Core_PC1_to_RF_SI_gamma_{gamma:g}_net"
    candidate_strategy_returns[name] = strat_rf.rename(name)
    candidate_strategy_diags[name] = diag_rf

benchmark_returns = {
    "Always_PC1_net": core_pc1.rename("Always_PC1_net"),
    "QQQ_buy_hold": B_ret.reindex(core_pc1.index).rename("QQQ_buy_hold"),
    "Satellite_tangency": satellite_basket_ret_tan.reindex(core_pc1.index).rename("Satellite_tangency"),
    "RF_proxy": rf_core.reindex(core_pc1.index).rename("RF_proxy"),
}
all_returns = {**benchmark_returns, **candidate_strategy_returns}
strategy_panel = pd.concat(all_returns, axis=1).dropna(how="all")

if isinstance(strategy_panel.columns, pd.MultiIndex):
    strategy_panel.columns = strategy_panel.columns.get_level_values(0)

print("Overall strategy metrics on tradable range")
display(
    metrics_table_for_series(
        {k: strategy_panel[k].dropna() for k in strategy_panel.columns},
        rf_daily=rf_core.reindex(strategy_panel.index).ffill().fillna(0.0),
    )
)

periods = {
    "validation": (sat_val_start, sat_val_end),
    "frozen_final_oos": (sat_final_start, strategy_panel.index.max()),
}
for pname, (a, b) in periods.items():
    sub = strategy_panel.loc[(strategy_panel.index >= a) & (strategy_panel.index <= b)].dropna(how="all")
    rf_sub = rf_core.reindex(sub.index).ffill().fillna(0.0)
    print(f"\n{pname.upper()} metrics: {pd.Timestamp(a).date()} -> {pd.Timestamp(b).date()} | n={len(sub)}")
    display(metrics_table_for_series({c: sub[c].dropna() for c in sub.columns}, rf_daily=rf_sub))

val_results = []
for name in candidate_strategy_returns.keys():
    if name not in strategy_panel.columns:
        continue
    r = strategy_panel[name].dropna()
    r_val = r.loc[(r.index >= sat_val_start) & (r.index <= sat_val_end)]
    if len(r_val) < 50:
        continue
    rf_val = rf_core.reindex(r_val.index).ffill().fillna(0.0)
    mets = perf_metrics(r_val, rf_daily=rf_val)
    mets["strategy"] = name
    val_results.append(mets)

if len(val_results) == 0:
    raise ValueError("No valid candidate strategies available in validation window.")

val_rank = (
    pd.DataFrame(val_results)
    .set_index("strategy")
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["Martin"])
    .sort_values("Martin", ascending=False)
)

if len(val_rank) == 0:
    raise ValueError("Validation ranking is empty after filtering invalid Martin ratios.")

print("\nValidation ranking of SI allocation variants by Martin ratio:")
display(val_rank[["CAGR", "ShR", "Martin", "MxDD", "Ulcer", "n"]])

top_k = min(CORE_SAT_VALIDATION_TOP_K, len(val_rank))
top_val = val_rank.head(top_k).copy()

BEST_SI_STRATEGY = top_val.sort_values(["CAGR", "ShR"], ascending=False).index[0]
print(f"Selected validation-frozen SI strategy (top {top_k} by Martin, then highest CAGR):", BEST_SI_STRATEGY)

plot_cols = ["Always_PC1_net", "QQQ_buy_hold", "Satellite_tangency", BEST_SI_STRATEGY]
plot_cols = [c for c in plot_cols if c in strategy_panel.columns]
plot_df = strategy_panel[plot_cols].dropna(how="all")

if len(plot_df) > 0:
    ((1.0 + plot_df.fillna(0.0)).cumprod()).plot(
        title="Core-satellite SI allocation vs PC1 / QQQ / tangency satellite"
    )
    plt.show()

chosen_diag = candidate_strategy_diags[BEST_SI_STRATEGY]
print("\nChosen SI strategy allocation/cost diagnostics:")
diag_out = {
    "mean_SI_lag": chosen_diag["SI_lag"].mean() if "SI_lag" in chosen_diag.columns else np.nan,
    "p75_SI_lag": chosen_diag["SI_lag"].quantile(0.75) if "SI_lag" in chosen_diag.columns else np.nan,
    "max_SI_lag": chosen_diag["SI_lag"].max() if "SI_lag" in chosen_diag.columns else np.nan,
    "n_allocation_changes": int((chosen_diag["overlay_turnover"] > 1e-12).sum()),
    "avg_overlay_turnover": chosen_diag["overlay_turnover"].mean(),
    "total_overlay_cost_bps": 1e4 * chosen_diag["overlay_cost"].sum(),
    "avg_daily_overlay_cost_bps": 1e4 * chosen_diag["overlay_cost"].mean(),
}
if "outer_gamma" in chosen_diag.columns:
    diag_out["outer_gamma"] = chosen_diag["outer_gamma"].iloc[0]
if "rf_gamma" in chosen_diag.columns:
    diag_out["rf_gamma"] = chosen_diag["rf_gamma"].iloc[0]
if "w_core" in chosen_diag.columns:
    diag_out["avg_w_core"] = chosen_diag["w_core"].mean()
if "w_tan_total" in chosen_diag.columns:
    diag_out["avg_w_tan_total"] = chosen_diag["w_tan_total"].mean()
if "w_rf_endpoint" in chosen_diag.columns:
    diag_out["avg_w_rf_endpoint"] = chosen_diag["w_rf_endpoint"].mean()
display(pd.Series(diag_out).to_frame("value"))

final_cols = [BEST_SI_STRATEGY, "Always_PC1_net", "QQQ_buy_hold"]
final_df = strategy_panel[final_cols].loc[strategy_panel.index >= sat_final_start].dropna(how="any")

print("\nPaired stationary-bootstrap differences in frozen final OOS: selected SI strategy minus benchmarks")
if len(final_df) >= 50:
    boot_diff_final = paired_bootstrap_metric_diffs(
        df_returns=final_df,
        target=BEST_SI_STRATEGY,
        benchmarks=["Always_PC1_net", "QQQ_buy_hold"],
        rf_daily=rf_core.reindex(final_df.index).ffill().fillna(0.0),
        n_boot=SAT_BOOT_N,
        avg_block_len=SAT_BOOT_AVG_BLOCK,
        seed=11,
    )
    display(boot_diff_final)
else:
    boot_diff_final = pd.DataFrame()
    print(f"Skipped paired bootstrap: not enough final OOS observations. n={len(final_df)}")

# ------------------------------
# Safe output collection
# ------------------------------
screen_results_out = globals().get(
    "screen_results",
    SATELLITE_BUILD_OBJECTS.get("screen_results", None) if isinstance(SATELLITE_BUILD_OBJECTS, dict) else None
)

selected_assets_out = globals().get(
    "selected_assets",
    SATELLITE_BUILD_OBJECTS.get("selected_assets", None) if isinstance(SATELLITE_BUILD_OBJECTS, dict) else None
)

sat_w_tan_out = globals().get(
    "sat_w_tan",
    SATELLITE_BUILD_OBJECTS.get("satellite_tangency_weights", None) if isinstance(SATELLITE_BUILD_OBJECTS, dict) else None
)

CORE_SATELLITE_RESULTS = {
    "satellite_risky_screen_results": screen_results_out,
    "selected_risky_assets": selected_assets_out,
    "satellite_tangency_weights": sat_w_tan_out,
    "strategy_panel": strategy_panel,
    "strategy_diags": candidate_strategy_diags,
    "validation_ranking": val_rank,
    "best_strategy": BEST_SI_STRATEGY,
    "final_oos_bootstrap": boot_diff_final,
    "split": SATELLITE_BUILD_OBJECTS.get("split", None) if isinstance(SATELLITE_BUILD_OBJECTS, dict) else None,
}

## 8) Roadmap / TODO

- refine the IV overlay beyond level / change / z-score (for example, IV term structure, skew, or richer cross-risk-premia state variables);
- test alternative downside targets (for example, true forward maximum drawdown vs worst return from entry) and alternative event thresholds / maturities;
- add explicit **overlay trading-cost** modelling on top of the current base series, which is already net of spread-based costs;
- extend execution modelling beyond spread-based costs (slippage / impact proxies and flat-cost robustness checks);
- replicate the framework across additional regions (EU / UK / JP) with a unified regime lens;
- run broader sensitivity checks on regime definition, bootstrap block length, `k`, covariance shrinkage, and overlay exposure maps.
